#Imports

In [ ]:
!pip install -q timm==1.0.19 gdown

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.1 MB/s eta 0:00:00


In [ ]:
#import random
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import timm
import json

from PIL import Image
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix,f1_score
from torchvision.transforms import InterpolationMode
import math
#from torchvision.transforms.functional import rgb_to_grayscale

#Globals

In [ ]:
category_mapping = {
  0: "original",
  1: "redigital",
  2: "transfer"
}

class_mapping = {
  0: "real",
  1: "ai"
}

CATEGORY_WEIGHT = 1.0 #Multi-Head weight loss for category transformations ("original","redigital","transfer")
CLASS_WEIGHT = 1.0 #Multi-Head weight loss for class real/fake
batch_size = 32
lr = 3e-4  #Learning Rate for Training process
num_epochs = 20
LAZY_LOAD = True #USE FALSE ONLY IF YOU HAVE AT LEAST 20 GB OF GPU MEMORY IN ORDER TO SPEED-UP TRAINING!
Selected_seed = 4444
loss_fn = nn.CrossEntropyLoss() #Type of loss used for AI/Real class classification and category transformation classification
DRCT_CHECKPOINT_PATH = '/content/last_acc0.9991.pth' #where the DRCT checkpoint will be locally saved
#@markdown Insert here the path of the folder where you plan to place the Archive of the Dataset ([download instruction here](#scrollTo=0tz9Ebt6razA&line=2&uniqifier=1)).
FOLDER_PATH = "/content/drive/MyDrive/Computer Vision/Project/" #@param {type:"string"}
ARCHIVE_NAME= "RRDataset_subset_final.tar"
DATASET_SUBSET_ARCHIVE_ABSOLUTE_PATH = os.path.join(FOLDER_PATH,ARCHIVE_NAME)
LOCAL_DATASET = "/content/RRDataset_subset_final" # where the subset of the dataset will be locally extracted

#@markdown Do you want to save the training results in drive?
save_in_drive = False #@param {type:"boolean"}
path_to_save_results_on_drive= '/content/drive/MyDrive/Computer Vision/Project/MultiHead_DFT_trial/' #Where to save training results on drive


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


#Utils

In [ ]:
#@title Replicability Settings
def set_seed(seed = 4444):
  """
  With this function we set the seed for all the libraries in order to make the experiments replicable

  """
  os.environ["PYTHONHASHSEED"] = str(seed)
  generator = torch.Generator()
  generator.manual_seed(seed)
  torch.manual_seed(seed)
  np.random.seed(seed)
  torch.use_deterministic_algorithms(True)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
  return seed, generator

SEED, GENERATOR= set_seed(Selected_seed)

In [ ]:
#@title Radial Map
def create_normalized_radial_map(height = 224,width = 224,device = "cpu"):
  """
  Create a normalized radial map for each pixel of an image of dimensions [H,W] (default H=224, W=224) w.r.t. the center.
  Each pixel of this new image has the value of the distance from the center of the image (measured in pixels). At the end, these distances are normalized between 0 and 1.
  The pixel in the center has value 0.
  The pixels in the corners have value 1.
  It can be computed once and then used for every Images of shape [H,W]

  This function returns a tensor of shape [1,1,H,W].
  """


  # Get all the possible coordinates (both x and y) relative to the center of the Image of shape [H,W], and save them in two different tensors.
  y = (torch.arange(height,device=device,dtype=torch.float32)- height // 2)

  x = (torch.arange(width,device=device,dtype=torch.float32)- width // 2)

  # Build two different 2D Grid of size [H,W], where each cell of y_pixel_coordinates (x_pixel_coordinates) has the y (x) coordinate w.r.t the center of that pixel
  # Thus the point in the center of image has coordinates (0,0)
  y_pixel_coordinates, x_pixel_coordinates = torch.meshgrid(y,x,indexing="ij")

  # Compute the Radial Map through the Euclidean distance of each pixel from the center
  radial_map = torch.sqrt(x_pixel_coordinates.square() + y_pixel_coordinates.square())

  # Normalization of the distance from the center in the interval [0,1]
  max_radius = radial_map.max()
  radial_map = radial_map / max_radius

  # Add batch size and channel to the radial map:
  # [H,W] -> [1,1,H,W]
  radial_map = radial_map.unsqueeze(0).unsqueeze(0)

  return radial_map

In [ ]:
#@title DFT amplitude
def compute_log_dft_amplitude(batch_rgb):
  """
  This function computes the DFT (log-)amplitude of a batch of RGB images [B, 3, H, W].
  """

  rgb = batch_rgb.float()

  # Convert each RGB Image to Grayscale
  red = rgb[:, 0:1]
  green = rgb[:, 1:2]
  blue = rgb[:, 2:3]

  grayscale = (
    0.299 * red
    + 0.587 * green
    + 0.114 * blue)

  # Bidimensional DFT
  spectrum = torch.fft.fft2(
    grayscale,
    dim=(-2, -1), #On which dimension
    norm="ortho" #Which type of normalization
  )

  # Shifting low frequencies in the center
  spectrum = torch.fft.fftshift(spectrum,dim=(-2, -1))

  # amplitude of the spectrum
  amplitude = torch.abs(spectrum)

  # Compress the amplitude at frequency zero
  amplitude = torch.log1p(amplitude)

  # Normalization for mean and std of the frequencies in the same DFT amplitude image
  mean = amplitude.mean(
    dim=(-2, -1),
    keepdim=True)

  std = amplitude.std(
    dim=(-2, -1),
    keepdim=True,
    unbiased=False)

  amplitude = (amplitude - mean) / std.clamp_min(1e-5) #No division by zero

  # Return tensor of shape [B, 1, H, W].
  return amplitude

In [ ]:
#@title Calculate metrics Single Head Real/Fake
def calculate_metrics(y_true_list, y_pred_list, metrics):
  accuracy = accuracy_score(y_true=y_true_list, y_pred=y_pred_list)
  confusion_mat = confusion_matrix(y_true=y_true_list, y_pred=y_pred_list,labels=[0, 1])
  f1 = f1_score(y_true=y_true_list, y_pred=y_pred_list,average="macro")


  metrics['accuracy'].append(float(accuracy))
  metrics['confusion_mat'].append(confusion_mat.tolist())
  metrics['f1'].append(float(f1))

  return accuracy,confusion_mat,f1

In [ ]:
#@title Calculate Metrics Multi Head
def calculate_metrics_multihead(y_true_list, y_pred_list, metrics,task):
  if task == "class":
    labels=[0, 1]
  elif task == "category":
    labels=[0, 1, 2]
  else:
    print("Error Task!")
    return

  accuracy = accuracy_score(y_true=y_true_list, y_pred=y_pred_list)
  confusion_mat = confusion_matrix(y_true=y_true_list, y_pred=y_pred_list,labels=labels)
  f1 = f1_score(y_true=y_true_list, y_pred=y_pred_list,average="macro",)

  metrics['accuracy_'+task].append(float(accuracy))
  metrics['confusion_mat_'+task].append(confusion_mat.tolist())
  metrics['f1_'+task].append(float(f1))

  return accuracy,confusion_mat,f1

In [ ]:
#@title Configure phase 1 of fine-tuning MultiHead
def configure_fine_tuning_multihead_phase_1(model):

  # Block the training for all parts of the model
  for parameter in model.parameters():
    parameter.requires_grad = False

  # Unlock the training only for the new head for detecting category transformations
  for parameter in model.fc_category.parameters():
    parameter.requires_grad = True

  optimizer = torch.optim.AdamW(
    model.fc_category.parameters(),
    lr=1e-3,
    weight_decay=1e-4)

  return optimizer

In [ ]:
#@title Configure phase 2 of fine-tuning MultiHead
def configure_fine_tuning_multihead_phase_2(model):

  """
  Unlock the training for all parts of the model, and assign different learning rates to each part
  """
  for parameter in model.parameters():
    parameter.requires_grad = True

  parameter_groups = []

  if isinstance(model, DRCTConvB_DFT_MultiHead):
    parameter_groups.append(
      {
        "params": model.dft_stem.parameters(),
        "lr": 1e-3
      })

  parameter_groups.extend(
    [
      {
        "params": model.model.parameters(),
        "lr": 1e-3
      },
      {
        "params": model.fc.parameters(),
        "lr": 3e-5
      },
      {
        "params": model.fc_category.parameters(),
        "lr": 3e-4
      }])

  optimizer = torch.optim.AdamW(
    parameter_groups,
    weight_decay=1e-4)

  return optimizer

#Data

**Download the archive subset dataset from** https://drive.google.com/file/d/1Y9WJSk2nGYXYGO9T6PcgerI0cK-aGbHn/view?usp=sharing
**and place it on the folder specified by the [FOLDER_PATH](#scrollTo=1qOt_vHa75kz&line=21&uniqifier=1) variable**

The original RRDataset is uploaded here: https://drive.google.com/drive/folders/1fTFIHXxDNseudhx9EJI-QoA0ZtzvGv8O?usp=sharing

In [ ]:
#Connection to google drive is optional
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
#@title Move the subset archive in the /content/
!rsync -ah --info=progress2 \
"{DATASET_SUBSET_ARCHIVE_ABSOLUTE_PATH}" \
"/content/"

          3.55G 100%   29.32MB/s    0:01:55 (xfr#1, to-chk=0/1)


In [ ]:
#@title Extract the Dataset in the /content/
!tar -xf "/content/RRDataset_subset_final.tar" -C "/content/"

In [ ]:
#@title Split dataset

#From the downloaded and extracted dataset open the file "subset_list.csv"
csv_path_on_local = os.path.join(LOCAL_DATASET, "subset_list.csv")
complete_dataframe = pd.read_csv(csv_path_on_local)

"""
The subset_list.csv file is used to reconstruct and navigate the subset of RRDataset previously created (maintaning the same folder structure of the original datatset).
It has the following structure:
path_on_drive: the absolute path of the image when the subset was created in my drive
category_transformation: which transformation the image belongs to
class: if the image is real or AI generated
label_AI: 0 for real, 1 for fake (reduntant)
"""

# Take only the name of the images (onyl the last part of the path)
complete_dataframe["image_name"] = complete_dataframe["path_on_drive"].apply(os.path.basename)

# Reconstruct, for each image, the absolute path in the local machine
complete_dataframe["local_path"] = complete_dataframe.apply(
  lambda row: os.path.join(
    LOCAL_DATASET,
    row["category_transformation"],
    row["class"],
    row["image_name"]),
  axis=1)

"""
In order to make a subset balanced split "across real/fake classes and transformation categories",
we need to divide images in 6 big families, and using them in the stratification split rule:
- original/ai
- original/real
- redigital/ai
- redigital/real
- transfer/ai
- transfer/real
"""

# Stratification for category and class
complete_dataframe["complete_family"] = (
  complete_dataframe["category_transformation"].astype(str)
  + "_"
  + complete_dataframe["class"].astype(str))

# 70% training set, 30% validation+test sets
train_dataframe, temp_df = train_test_split(
  complete_dataframe,
  test_size=0.30,
  random_state=SEED,
  stratify=complete_dataframe["complete_family"])

# 15% validation set, 15% test set
val_dataframe, test_dataframe = train_test_split(
  temp_df,
  test_size=0.50,
  random_state=SEED,
  stratify=temp_df["complete_family"])

train_num_samples_by_category_and_class = (
  train_dataframe
  .groupby(["category_transformation", "class"])
  .agg(num_samples=("class", "size"))
  .reset_index())

val_num_samples_by_category_and_class = (
  val_dataframe
  .groupby(["category_transformation", "class"])
  .agg(num_samples=("class", "size"))
  .reset_index())

test_num_samples_by_category_and_class = (
  test_dataframe
  .groupby(["category_transformation", "class"])
  .agg(num_samples=("class", "size"))
  .reset_index())

print(train_num_samples_by_category_and_class.to_string(index=False))
print(val_num_samples_by_category_and_class.to_string(index=False))
print(test_num_samples_by_category_and_class.to_string(index=False))

print("Total number of Training Images:", len(train_dataframe))
print("Total number of Validation Images:", len(val_dataframe))
print("Total number of Test Images:", len(test_dataframe))

category_transformation class  num_samples
               original    ai         1050
               original  real         1050
              redigital    ai         1050
              redigital  real         1050
               transfer    ai         1050
               transfer  real         1050
category_transformation class  num_samples
               original    ai          225
               original  real          225
              redigital    ai          225
              redigital  real          225
               transfer    ai          225
               transfer  real          225
category_transformation class  num_samples
               original    ai          225
               original  real          225
              redigital    ai          225
              redigital  real          225
               transfer    ai          225
               transfer  real          225
Total number of Training Images: 6300
Total number of Validation Images: 1350
Total number of Tes

In [ ]:
#@title Class Dataset and Creation of DataLoaders

"""
We have two types of image-loading processes:
1) LAZY LOADING: We re-open each image and re-compute the DFT amplitude every time they are needed (slower in the training but it is necessary when we have low GPU memory)
2) EAGER LOADING: We open each image and compute the DFT amplitude only in the initialization of RRDataset and we maintain them in the GPU memory (faster in the training but 20 GB of GPU Memory required)

In the Dataframe all the informations neeeded for the image are registered (local file system location, category transformation, class etc.)
"""
if LAZY_LOAD:
  class RRDataset_subset(Dataset):

    def __init__(self, dataframe):
      self.dataframe = dataframe.reset_index(drop=True)

      # DRCT Transformation Part 1(without augmentation or blurring)
      self.pre_dft_transform = transforms.Compose([
        transforms.Resize(
          size=256,
          interpolation=InterpolationMode.BICUBIC,
          antialias=True),
        transforms.CenterCrop(size=(224, 224)),
        transforms.ToTensor()])
      # DRCT Transformation Part 2 (after computing the DFT)
      # ImageNet Normalization
      self.rgb_normalize_transform = transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225))
      print("Lazy Loading")

    def __len__(self):
      return len(self.dataframe)

    def __getitem__(self, index):
      row = self.dataframe.iloc[index]

      image_path = row["local_path"]

      class_label = int(row["label_AI"]) #Label of the class (0: Real, 1:AI)

      image = Image.open(image_path).convert("RGB")

      rgb=self.pre_dft_transform(image)

      #Before the normalization of the RGB Image, we need to compute the DFT. And only after that continue with normalization
      dft_amplitude = compute_log_dft_amplitude(rgb.unsqueeze(0)).squeeze(0)

      #DRCT Transformation Part 2
      rgb_normalized= self.rgb_normalize_transform(rgb)

      class_label = torch.tensor(class_label, dtype=torch.long)


      # Mapping the category transformation to 0,1,2
      if row["category_transformation"] == "original":
        category_label = torch.tensor(0, dtype=torch.long)
      if row["category_transformation"] == "redigital":
        category_label = torch.tensor(1, dtype=torch.long)
      if row["category_transformation"] == "transfer":
        category_label = torch.tensor(2, dtype=torch.long)

      return rgb_normalized,dft_amplitude, class_label , category_label

  train_dataset = RRDataset_subset(train_dataframe)

  val_dataset = RRDataset_subset(val_dataframe)

  test_dataset = RRDataset_subset(test_dataframe)

  train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    drop_last= False,
    generator=GENERATOR)

  val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True)

  test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True)
  print("Lazy Loading complete")
else:
  class RRDataset_subset(Dataset):

    def __init__(self, dataframe, device = "cuda"):
      self.dataframe = dataframe.reset_index(drop=True)

      # DRCT Transformation Part 1(without augmentation or blurring)
      self.pre_dft_transform = transforms.Compose([
        transforms.Resize(
          size=256,
          interpolation=InterpolationMode.BICUBIC,
          antialias=True),
        transforms.CenterCrop(size=(224, 224)),
        transforms.ToTensor()])

      # DRCT Transformation Part 2 (after computing the DFT)
      # ImageNet Normalization
      self.rgb_normalize_transform = transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225))

      print("Eager Loading")

      self.pre_processed_data = []
      for index in tqdm(range(len(self.dataframe)), desc="Loading..."):
        row = self.dataframe.iloc[index]
        image_path = row["local_path"]
        label_val = int(row["label_AI"]) #Label of the class (0: Real, 1:AI)

        image = Image.open(image_path).convert("RGB")

        rgb = self.pre_dft_transform(image).to(device)

        #Before the normalization of the RGB Image, we need to compute the DFT. And only after that continue with normalization
        dft_amplitude = compute_log_dft_amplitude(rgb.unsqueeze(0)).squeeze(0)

        #DRCT Transformation Part 2
        rgb_normalized = self.rgb_normalize_transform(rgb)

        class_label = torch.tensor(label_val, dtype=torch.long).to(device)

        # Mapping the category transformation to 0,1,2
        if row["category_transformation"] == "original":
          category_label = torch.tensor(0, dtype=torch.long).to(device)
        if row["category_transformation"] == "redigital":
          category_label = torch.tensor(1, dtype=torch.long).to(device)
        if row["category_transformation"] == "transfer":
          category_label = torch.tensor(2, dtype=torch.long).to(device)

        self.pre_processed_data.append((
          rgb_normalized,
          dft_amplitude.to(device),
          class_label,
          category_label))

    def __len__(self):
      return len(self.pre_processed_data)

    def __getitem__(self, index):
      return self.pre_processed_data[index]


  train_dataset = RRDataset_subset(train_dataframe,device=device)

  val_dataset = RRDataset_subset(val_dataframe,device=device)

  test_dataset = RRDataset_subset(test_dataframe,device=device)

  train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    #persistent_workers=True,
    drop_last= False,
    generator=GENERATOR)

  val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    #persistent_workers=True
    )

  test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=False)

  print("Eager Loading complete")

Lazy Loading
Lazy Loading
Lazy Loading
Lazy Loading complete


#Network

In [ ]:
#@title Download the DRCTConvB Checkpoint in /content/ as last_acc0.9991.pth
!gdown --fuzzy "https://drive.google.com/file/d/1LXLXAlsomU5o3AjauINmOlokSvJIGE0q/view?usp=sharing"

Downloading...
From (original): https://drive.google.com/uc?id=1LXLXAlsomU5o3AjauINmOlokSvJIGE0q
From (redirected): https://drive.google.com/uc?id=1LXLXAlsomU5o3AjauINmOlokSvJIGE0q&confirm=t&uuid=7f0ea0b9-c077-4397-b979-21e8bf37aa02
To: /content/last_acc0.9991.pth
100% 355M/355M [00:09<00:00, 39.3MB/s]


In [ ]:
#@title Download my personal DRCTConvB Checkpoint in /content/ as "best_model_accuracy_base.pt"
!gdown --fuzzy "https://drive.google.com/file/d/18BRyXCF1kSpfi2IGsXEI7j5fRCinoO-s/view?usp=sharing"

In [ ]:
#@title Download my personal DRCTConvB_DFT Checkpoint in /content/ as "best_model_accuracy_DFT.pt"
!gdown --fuzzy "https://drive.google.com/file/d/1kK0usJh56bbYRQF_q6rKHMVO0IYqv4uT/view?usp=sharing"

Downloading...
From (original): https://drive.google.com/uc?id=1kK0usJh56bbYRQF_q6rKHMVO0IYqv4uT
From (redirected): https://drive.google.com/uc?id=1kK0usJh56bbYRQF_q6rKHMVO0IYqv4uT&confirm=t&uuid=9b04029d-5cc9-4b5b-bea6-42ecd3ae2f66
To: /content/best_model_accuracy_DFT.pt
100% 355M/355M [00:04<00:00, 81.4MB/s]


In [ ]:
#@title DRCTConvB SINGLE HEAD
class DRCTConvB(nn.Module):
  def __init__(
    self,
    checkpoint_path = None,
    map_location = "cpu",
    strict = True,
    device = "cpu"):
    super().__init__()
    """
    If it is given the DRCT_checkpoint_path, this class will load the state dict.
    Otherwise the class will initialize randomly the model
    """

    self.model = timm.create_model("convnext_base_in22k",pretrained=False)

    """
    In the original ConvNeXt_Base timm this is the final part of the model:

    head.fc:
       1024 features -> ImageNet classes


    DRCT modified the head.fc output with:
       1024 features -> 1024 embedding_size
    and then added at the end a new head MLP called "fc" as:
       1024 embedding_size -> 2 classes (real/AI)
    """
    convnext_base_head_feature_dim = self.model.head.fc.in_features

    self.model.head.fc = nn.Linear(in_features=convnext_base_head_feature_dim,out_features=1024)

    # Binary Classificator DRCT:
    # embedding 1024 -> 2 logits.
    self.fc = nn.Linear(in_features=1024,out_features=2)

    if checkpoint_path is not None:
      state_dict = torch.load(checkpoint_path,map_location=map_location,weights_only=True)
      self.load_state_dict(state_dict,strict=strict)
      print("Checkpoint DRCT loaded correctly.")

  def forward(self,x):
    # [B, 1024]
    features = self.model(x)
    # [B, 2]
    logits = self.fc(features)

    return logits

# model = DRCTConvB(checkpoint_path=DRCT_CHECKPOINT_PATH,device=device)
# model = model.to(device)

In [ ]:
#@title DRCTConvB_DFT SINGLE HEAD
class DRCTConvB_DFT(nn.Module):
  def __init__(
    self,
    checkpoint_path = None,
    map_location = "cpu",
    strict = True,
    device = "cpu"):
    super().__init__()

    """
    ############# THE NEW PROPOSED SINGLEHEAD MODEL WITH DFT STEM ###############
    If it is given the DRCT_checkpoint_path, this class will load the state dict before creating the new initial parallel DFT STEM (initialized to produce feature maps with all zeros).
    Otherwise the class will initialize randomly the model, apart from DFT STEM that will be initialized to produce feature maps with all zeros.
    """

    radial_normalized_map = create_normalized_radial_map(height=224,width=224,device=device)

    radial_normalized_map = radial_normalized_map.to(device)

    self.register_buffer("radial_map",radial_normalized_map,persistent=False)

    self.model = timm.create_model("convnext_base_in22k",pretrained=False)

    """
    In the original ConvNeXt_Base timm this is the final part of the model:

    head.fc:
       1024 features -> ImageNet classes


    DRCT modified the head.fc output with:
       1024 features -> 1024 embedding_size
    and then added at the end a new head MLP called "fc" as:
       1024 embedding_size -> 2 classes (real/AI)
    """
    convnext_base_head_feature_dim = self.model.head.fc.in_features

    self.model.head.fc = nn.Linear(in_features=convnext_base_head_feature_dim,out_features=1024)

    # Binary Classificator DRCT:
    # embedding 1024 -> 2 logits.
    self.fc = nn.Linear(in_features=1024,out_features=2)

    if checkpoint_path is not None:
      state_dict = torch.load(checkpoint_path,map_location=map_location,weights_only=True)
      self.load_state_dict(state_dict,strict=strict)
      print("Checkpoint DRCT loaded correctly.")

    # ---------------------------------------------------------
    # PARALLEL DFT STEM
    # ---------------------------------------------------------

    # Use the same structure of the RGB STEM of the original model:
    #
    # Sequential(
    #     Conv2d(3, 128, kernel_size=4, stride=4),
    #     LayerNorm2d(128)
    # )
    self.dft_stem = copy.deepcopy(self.model.stem) #Sequential(Conv2d(3, 128, kernel_size=4, stride=4),LayerNorm2d(128))

    rgb_stem_conv_2D = self.model.stem[0] # Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))

    self.dft_stem[0] = nn.Conv2d(
      in_channels=2, # DFT amplitude + radial map = 2 channels
      out_channels=rgb_stem_conv_2D.out_channels,
      kernel_size=rgb_stem_conv_2D.kernel_size,
      stride=rgb_stem_conv_2D.stride,
      padding=rgb_stem_conv_2D.padding,
      bias=rgb_stem_conv_2D.bias is not None)

    # The new branch DFT initially produces feature maps with all zeros
    nn.init.zeros_(self.dft_stem[0].weight)

    if self.dft_stem[0].bias is not None:
      nn.init.zeros_(self.dft_stem[0].bias)

    nn.init.ones_(self.dft_stem[1].weight)
    nn.init.zeros_(self.dft_stem[1].bias)

  def forward(self,rgb,dft):
    """
    Overlap in the channels the DFT amplitude with the corresponding Radial Map

    dft:        [B, 1, H, W]
    radial_map: [B, 1, H, W]

    dft_input:  [B, 2, H, W]
    """
    batch_size = dft.shape[0]

    # Radial Map extended to each DFT amplitude in the batch:
    # [1, 1, H, W] -> [B, 1, H, W]
    radial_map = self.radial_map.expand(batch_size,-1,-1,-1)

    # Overlap the DFT amplitude with the corresponding Radial Map
    dft_input = torch.cat([dft, radial_map],dim=1)

    # Both the two STEMs produce these feature maps:
    # [B, 128, H/4, W/4]
    rgb_features = self.model.stem(rgb)
    dft_features = self.dft_stem(dft_input)

    # Fusion of the feature maps produced by the two STEMs through SUM
    x = rgb_features + dft_features

    x = self.model.stages(x)
    x = self.model.norm_pre(x)

    # [B, 1024]
    features = self.model.head(x)
    # [B, 2]
    logits = self.fc(features)

    return logits

# model = DRCTConvB_DFT(checkpoint_path=DRCT_CHECKPOINT_PATH,device=device)
# model=model.to(device)

In [ ]:
#@title DRCTConvB MULTI HEAD
class DRCTConvB_MultiHead(nn.Module):
  def __init__(
    self,
    checkpoint_path = None,
    map_location = "cpu",
    strict = True,
    device = "cpu"):
    super().__init__()

    """
    If it is given the DRCT_checkpoint_path, this class will load the state dict.
    Otherwise the class will initialize randomly the model
    """

    self.model = timm.create_model("convnext_base_in22k",pretrained=False)

    """
    In the original ConvNeXt_Base timm this is the final part of the model:

    head.fc:
       1024 features -> ImageNet classes


    DRCT modified the head.fc output with:
       1024 features -> 1024 embedding_size
    and then added at the end a new head MLP called "fc" as:
       1024 embedding_size -> 2 classes (real/AI)
    """
    convnext_base_head_feature_dim = self.model.head.fc.in_features

    self.model.head.fc = nn.Linear(in_features=convnext_base_head_feature_dim,out_features=1024)

    # Binary Classificator DRCT:
    # embedding 1024 -> 2 logits.
    self.fc = nn.Linear(in_features=1024,out_features=2)

    if checkpoint_path is not None:
      state_dict = torch.load(checkpoint_path,map_location=map_location,weights_only=True)
      self.load_state_dict(state_dict,strict=strict)
      print("Checkpoint loaded correctly.")


    # CREATE AND INITIALIZE THE SECOND HEAD FOR THE CATEGORIES
    # This head will take as input the same features of self.fc
    self.fc_category = nn.Linear(in_features=1024,out_features=3)
    nn.init.xavier_uniform_(self.fc_category.weight)
    nn.init.zeros_(self.fc_category.bias)

  def forward(self,x):
    # [B, 1024]
    features = self.model(x)
    # [B, 2]
    logits_class = self.fc(features)
    # [B, 3]
    logits_category = self.fc_category(features)

    return logits_class, logits_category

# model = DRCTConvB_MultiHead(
#     checkpoint_path='/content/drive/MyDrive/Computer Vision/Project/Real_AI_RGB_lr1e3/best_model_accuracy.pt',
#     device=device)
# model=model.to(device)

In [ ]:
#@title DRCTConvB_DFT MULTI HEAD
class DRCTConvB_DFT_MultiHead(nn.Module):
  def __init__(
    self,
    checkpoint_path = None,
    personal_checkpoint_path = None,
    map_location = "cpu",
    strict = True,
    device = "cpu"):
    super().__init__()

    """
    ############# THE NEW PROPOSED MULTIHEAD MODEL WITH DFT STEM ###############
    If it is given the DRCT_checkpoint_path, this class will load the state dict before creating the new initial parallel DFT STEM (initialized to produce feature maps with all zeros).
    Otherwise the class will initialize randomly the model, apart from DFT STEM that will be initialized to produce feature maps with all zeros.
    """

    radial_normalized_map = create_normalized_radial_map(height=224,width=224,device=device)

    radial_normalized_map = radial_normalized_map.to(device)

    self.register_buffer("radial_map",radial_normalized_map,persistent=False,)

    self.model = timm.create_model("convnext_base_in22k",pretrained=False)

    """
    In the original ConvNeXt_Base timm this is the final part of the model:

    head.fc:
       1024 features -> ImageNet classes


    DRCT modified the head.fc output with:
       1024 features -> 1024 embedding_size
    and then added at the end a new head MLP called "fc" as:
       1024 embedding_size -> 2 classes (real/AI)
    """

    convnext_base_head_feature_dim = self.model.head.fc.in_features

    self.model.head.fc = nn.Linear(in_features=convnext_base_head_feature_dim,out_features=1024)

    # Binary Classificator DRCT:
    # embedding 1024 -> 2 logits.
    self.fc = nn.Linear(in_features=1024,out_features=2)

    if checkpoint_path is not None:
      state_dict = torch.load(checkpoint_path,map_location=map_location,weights_only=True)
      self.load_state_dict(state_dict,strict=strict)
      print("DCRT Checkpoint loaded correctly.")

    # ---------------------------------------------------------
    # PARALLEL DFT STEM
    # ---------------------------------------------------------

    # Use the same structure of the RGB STEM of the original model:
    #
    # Sequential(
    #     Conv2d(3, 128, kernel_size=4, stride=4),
    #     LayerNorm2d(128)
    # )
    self.dft_stem = copy.deepcopy(self.model.stem) #Sequential(Conv2d(3, 128, kernel_size=4, stride=4),LayerNorm2d(128))

    rgb_stem_conv_2D = self.model.stem[0] # Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))

    self.dft_stem[0] = nn.Conv2d(
      in_channels=2, # DFT amplitude + radial map = 2 channels
      out_channels=rgb_stem_conv_2D.out_channels,
      kernel_size=rgb_stem_conv_2D.kernel_size,
      stride=rgb_stem_conv_2D.stride,
      padding=rgb_stem_conv_2D.padding,
      bias=rgb_stem_conv_2D.bias is not None)

    # The new branch DFT initially produces feature maps with all zeros
    nn.init.zeros_(self.dft_stem[0].weight)

    if self.dft_stem[0].bias is not None:
      nn.init.zeros_(self.dft_stem[0].bias)

    nn.init.ones_(self.dft_stem[1].weight)
    nn.init.zeros_(self.dft_stem[1].bias)

    #It is possible to load a pre-trained DRCTConvB_DFT SIGNLE HEAD
    if personal_checkpoint_path is not None:
      state_dict = torch.load(personal_checkpoint_path,map_location=map_location,weights_only=True)
      self.load_state_dict(state_dict,strict=strict)
      print("Personal checkpoint loaded correctly.")

    # CREATE AND INITIALIZE THE SECOND HEAD FOR THE CATEGORIES
    # This head will take as input the same features of self.fc
    self.fc_category = nn.Linear(in_features=1024,out_features=3)
    nn.init.xavier_uniform_(self.fc_category.weight)
    nn.init.zeros_(self.fc_category.bias)

  def forward(self,rgb,dft):
    """
    Overlap in the channels the DFT amplitude with the corresponding Radial Map

    dft:        [B, 1, H, W]
    radial_map: [B, 1, H, W]

    dft_input:  [B, 2, H, W]
    """

    batch_size = dft.shape[0]
    # Radial Map extended to each DFT amplitude in the batch:
    # [1, 1, H, W] -> [B, 1, H, W]
    radial_map = self.radial_map.expand(batch_size,-1,-1,-1)

    # Overlap the DFT amplitude with the corresponding Radial Map
    dft_input = torch.cat([dft, radial_map],dim=1)

    # Both the two STEMs produce these feature maps:
    # [B, 128, H/4, W/4]
    rgb_features = self.model.stem(rgb)
    dft_features = self.dft_stem(dft_input)

    # Fusion of the feature maps produced by the two STEMs through SUM
    x = rgb_features + dft_features

    x = self.model.stages(x)
    x = self.model.norm_pre(x)

    # [B, 1024]
    features = self.model.head(x)
    # [B, 2]
    logits_class = self.fc(features)

    # [B, 3]
    logits_category = self.fc_category(features)

    return logits_class, logits_category

# model = DRCTConvB_DFT_MultiHead(
#     personal_checkpoint_path='/content/drive/MyDrive/Computer Vision/Project/Real_AI_RGB_DFT_lr1e3/best_model_accuracy_terza.pt',
#     device=device
# )
# model=model.to(device)

#Disclaimer
**N.B.** I took inspiration and adapted the Train and Test Loop skeletons from one of my previous projects in which I worked and collaborated in: https://github.com/cybernetic-m/DAgger4Robotics

However I did significant changes in order to adapt those skeletons to this work

#Train

##Single Head

In [ ]:
#@title One epoch single head
def one_epoch_singlehead(dataloader, model, optimizer, loss_fn, device, validation = False):

  # Set the modality of model depending if it is in tranining or validation mode
  if validation:
    model.eval()
  else:
    model.train()

  epoch_loss = 0
  num_samples = 0
  y_true_list = []
  y_pred_list = []

  # Set the gradient modality (if in validation mode, it do not compute the gradient)
  grad_modality = torch.no_grad() if validation else torch.enable_grad()

  with grad_modality:
    for batch in tqdm(dataloader, desc="Validation Batches" if validation else "Training Batches"):
      rgb,dft, y_true, cats = batch
      if LAZY_LOAD:
        rgb = rgb.to(device,non_blocking=True)
        y_true = y_true.to(device,non_blocking=True)


      if not validation:
        optimizer.zero_grad()

      if isinstance(model,DRCTConvB):
        logits = model(rgb)
      elif isinstance(model,DRCTConvB_DFT):
        if LAZY_LOAD:
          dft = dft.to(device,non_blocking=True)
        logits = model(rgb,dft)
      else:
        print("Model not supported!")
      loss = loss_fn(logits, y_true)

      y_pred= logits.argmax(dim=1)
      y_pred_list.append(y_pred.cpu().detach())
      y_true_list.append(y_true.cpu().detach())

      batch_size = y_true.size(0)

      if not validation:
        loss.backward()
        optimizer.step()

      epoch_loss += loss.item() * batch_size
      num_samples += batch_size
    epoch_loss_avg = epoch_loss / num_samples
  return epoch_loss_avg, y_true_list, y_pred_list

In [ ]:
#@title Train Loop Single head
def train_singlehead(train_loader, val_loader, model, optimizer, loss_fn, num_epochs, device):

  epoch_loss = 0
  epoch_val_loss = 0
  best_vloss = 10000000 # Starting value of the best validation loss to save the model
  best_vaccuracy = 0.0 # Starting value of the best validation accuracy to save the model

  train_metrics = {
    'accuracy': [],
    'confusion_mat': [],
    'f1': [],
    'loss': [],
  }
  val_metrics = {
    'accuracy': [],
    'confusion_mat': [],
    'f1': [],
    'loss': [],
  }

  for epoch in range(num_epochs):
    print(f"EPOCH: {epoch+1}/{num_epochs}")


    epoch_loss, train_y_true_list, train_y_pred_list = one_epoch_singlehead(train_loader, model, optimizer, loss_fn, device)
    epoch_vloss, val_y_true_list, val_y_pred_list = one_epoch_singlehead(val_loader, model, optimizer, loss_fn, device, validation = True)

    train_y_true_tensor = torch.cat(train_y_true_list, dim=0)
    train_y_pred_tensor = torch.cat(train_y_pred_list, dim=0)
    val_y_true_tensor = torch.cat(val_y_true_list, dim=0)
    val_y_pred_tensor = torch.cat(val_y_pred_list, dim=0)

    train_y_true_numpy = train_y_true_tensor.numpy()
    train_y_pred_numpy = train_y_pred_tensor.numpy()
    val_y_true_numpy = val_y_true_tensor.numpy()
    val_y_pred_numpy = val_y_pred_tensor.numpy()

    # If the loss in the validation set is better in this epoch (w.r.t. the previous epochs), save the model
    if epoch_vloss < best_vloss:
      best_vloss = epoch_vloss
      torch.save(model.state_dict(), "best_model_loss.pt")
      print(f"Model saved as best_model_loss.pt")

    # Compute the metrics for the train and validation sets and append in the dictionary
    train_accuracy,train_confusion_mat,train_f1 = calculate_metrics(y_true_list = train_y_true_numpy, y_pred_list= train_y_pred_numpy, metrics = train_metrics)
    val_accuracy,val_confusion_mat, val_f1 = calculate_metrics(y_true_list = val_y_true_numpy, y_pred_list= val_y_pred_numpy, metrics = val_metrics)
    train_metrics['loss'].append(epoch_loss)
    val_metrics['loss'].append(epoch_vloss)

    # If the accuracy in the validation set is better in this epoch (w.r.t. the previous epochs) and with an acceptable loss, save the model
    if val_accuracy > best_vaccuracy:
      if epoch_vloss < ((1.6)* best_vloss):
        best_vaccuracy = val_accuracy
        torch.save(model.state_dict(), "best_model_accuracy.pt")
        print(f"Model saved as best_model_accuracy.pt")

    print(
      f"TRAIN\t\t"
      f"Loss: {epoch_loss:.4f}, "
      f"Accuracy: {train_accuracy:.4f}, "
      f"F1: {train_f1:.4f}, "
    )

    print(
      f"VALIDATION\t"
      f"Loss: {epoch_vloss:.4f}, "
      f"Accuracy: {val_accuracy:.4f}, "
      f"F1: {val_f1:.4f}, "
    )

    print("Train confusion matrix:")
    print(train_confusion_mat)

    print("Validation confusion matrix:")
    print(val_confusion_mat)

  # Save the metrics in json
  with open("train_metrics.json", "w") as f:
    json.dump(train_metrics, f, indent=4)
  with open("val_metrics.json", "w") as f:
    json.dump(val_metrics, f, indent=4)
  print("Metrics of training and validation set saved in .json files!")

In [ ]:
#@title TRAIN DRCTConvB SINGLE HEAD on classification Real/AI task
"""
Train DRCTConvB with a single head (Real/AI used in RRDataset)) to see
the results it can reach without the other head (Category)
"""
drct_rgb_core = DRCTConvB(
    checkpoint_path=DRCT_CHECKPOINT_PATH,
    device=device
)
drct_rgb_core=drct_rgb_core.to(device)
optimizer = torch.optim.Adam(drct_rgb_core.parameters(), lr=lr)
loss_fn = nn.CrossEntropyLoss()

train_singlehead(
    train_loader = train_loader,
    val_loader = val_loader,
    model = drct_rgb_core,
    optimizer = optimizer,
    loss_fn = loss_fn,
    num_epochs = num_epochs,
    device = device
)

In [ ]:
#@title TRAIN DRCTConvB_DFT SINGLE HEAD on classification Real/Ai task

"""
Train DRCTConvB_DFT with a single head (Real/AI) to see
the results it can reach without the other head (Category)
and if it performs better than the DRCTConvB
"""
drct_dft = DRCTConvB_DFT(
    checkpoint_path=DRCT_CHECKPOINT_PATH,
    device=device
)
drct_dft=drct_dft.to(device)
optimizer = torch.optim.Adam(drct_dft.parameters(), lr=lr)
loss_fn = nn.CrossEntropyLoss()

train_singlehead(
    train_loader = train_loader,
    val_loader = val_loader,
    model = drct_dft,
    optimizer = optimizer,
    loss_fn = loss_fn,
    num_epochs = num_epochs,
    device = device
)

##Multi Head

In [ ]:
#@title One Epoch Multi Head
def one_epoch_multihead(dataloader, model, optimizer, loss_fn_class,loss_fn_category,class_weight, category_weight, device, validation = False):

  # Set the modality of model depending if it is in tranining or validation mode
  if validation:
    model.eval()
  else:
    model.train()

  epoch_loss_sum = 0.0
  epoch_class_loss_sum = 0.0
  epoch_category_loss_sum = 0.0
  num_samples = 0
  class_true_list = []
  class_pred_list = []
  category_true_list = []
  category_pred_list = []

  # Set the gradient modality (if in validation mode, it do not compute the gradient)
  grad_modality = torch.no_grad() if validation else torch.enable_grad()

  with grad_modality:
    for batch in tqdm(dataloader, desc="Validation Batches" if validation else "Training Batches"):
      rgb,dft, class_true, category_true = batch
      if LAZY_LOAD:
        rgb = rgb.to(device,non_blocking=True)
        class_true = class_true.to(device,non_blocking=True)
        category_true = category_true.to(device,non_blocking=True)

      if not validation:
        optimizer.zero_grad()

      if isinstance(model,DRCTConvB_MultiHead):
        logits_class, logits_category = model(rgb)
      elif isinstance(model,DRCTConvB_DFT_MultiHead):
        if LAZY_LOAD:
          dft = dft.to(device,non_blocking=True)
        logits_class, logits_category = model(rgb,dft)
      else:
        print("Model not supported!")
      loss_class = loss_fn_class(logits_class, class_true) / math.log(2) #Normlization of the binary Real/AI classfication loss by the uniform-prediction log(2)
      loss_category = loss_fn_category(logits_category, category_true) / math.log(3) #Normlization of the three category transformation classfication loss by the uniform-prediction log(3)
      loss = ((class_weight*loss_class) + (category_weight * loss_category))/(class_weight + category_weight) #Weighted combined loss of class AI/Real loss and category transformation loss

      class_pred= logits_class.argmax(dim=1)
      class_true_list.append(class_true.cpu().detach())
      class_pred_list.append(class_pred.cpu().detach())

      category_pred= logits_category.argmax(dim=1)
      category_true_list.append(category_true.cpu().detach())
      category_pred_list.append(category_pred.cpu().detach())

      batch_size = class_true.size(0)
      num_samples += batch_size

      epoch_loss_sum += loss.item() * batch_size
      epoch_class_loss_sum += loss_class.item() * batch_size
      epoch_category_loss_sum += loss_category.item() * batch_size

      if not validation:
        loss.backward()
        optimizer.step()
    epoch_loss_avg = epoch_loss_sum / num_samples
    epoch_class_loss_avg = epoch_class_loss_sum / num_samples
    epoch_category_loss_avg = epoch_category_loss_sum / num_samples
  return epoch_loss_avg, class_true_list, class_pred_list, category_true_list, category_pred_list, epoch_class_loss_avg, epoch_category_loss_avg

In [ ]:
#@title Train Loop Multihead
def train_multihead(train_loader, val_loader, model, optimizer, loss_fn_class, loss_fn_category,class_weight, category_weight, num_epochs, device, path_to_save_drive = None):

  epoch_combined_loss = 0
  epoch_combined_vloss = 0
  best_vloss = 10000000 # Starting value of the best validation loss to save the model
  best_vaccuracy_class = 0.0 # Starting value of the best validation class AI/Real accuracy to save the model
  best_vaccuracy_category = 0.0 # Starting value of the best validation category transformation accuracy to save the model

  if save_in_drive and path_to_save_drive is not None:
    save_drive_dir = path_to_save_drive
    os.makedirs(save_drive_dir, exist_ok=True)

  train_metrics = {
    'accuracy_class': [],
    'f1_class' : [],
    'confusion_mat_class': [],
    'accuracy_category': [],
    'f1_category': [],
    'confusion_mat_category': [],
    'combined_loss': [],
    'class_loss': [],
    'category_loss': [],
  }
  val_metrics = {
    'accuracy_class': [],
    'f1_class' : [],
    'confusion_mat_class': [],
    'accuracy_category': [],
    'f1_category': [],
    'confusion_mat_category': [],
    'combined_loss': [],
    'class_loss': [],
    'category_loss': [],
  }

  for epoch in range(num_epochs):
    print(f"EPOCH: {epoch+1}/{num_epochs}")


    epoch_combined_loss, train_class_true_list, train_class_pred_list, train_category_true_list, train_category_pred_list,epoch_class_loss,epoch_category_loss = one_epoch_multihead(train_loader, model, optimizer, loss_fn_class,loss_fn_category,class_weight,category_weight, device)
    epoch_combined_vloss, val_class_true_list, val_class_pred_list, val_category_true_list, val_category_pred_list,val_epoch_class_loss,val_epoch_category_loss = one_epoch_multihead(val_loader, model, optimizer, loss_fn_class,loss_fn_category,class_weight, category_weight, device, validation = True)

    train_class_true_tensor = torch.cat(train_class_true_list, dim=0)
    train_class_pred_tensor = torch.cat(train_class_pred_list, dim=0)
    val_class_true_tensor = torch.cat(val_class_true_list, dim=0)
    val_class_pred_tensor = torch.cat(val_class_pred_list, dim=0)
    train_category_true_tensor = torch.cat(train_category_true_list, dim=0)
    train_category_pred_tensor = torch.cat(train_category_pred_list, dim=0)
    val_category_true_tensor = torch.cat(val_category_true_list, dim=0)
    val_category_pred_tensor = torch.cat(val_category_pred_list, dim=0)



    train_class_true_numpy = train_class_true_tensor.numpy()
    train_class_pred_numpy = train_class_pred_tensor.numpy()
    val_class_true_numpy = val_class_true_tensor.numpy()
    val_class_pred_numpy = val_class_pred_tensor.numpy()
    train_category_true_numpy = train_category_true_tensor.numpy()
    train_category_pred_numpy = train_category_pred_tensor.numpy()
    val_category_true_numpy = val_category_true_tensor.numpy()
    val_category_pred_numpy = val_category_pred_tensor.numpy()

    # If the combined loss in the validation set is better in this epoch (w.r.t. the previous epochs), save the model
    if epoch_combined_vloss < best_vloss:
      best_vloss = epoch_combined_vloss
      torch.save(model.state_dict(), "best_model_loss.pt")
      print(f"Model saved as best_model_loss.pt")
      if save_in_drive and path_to_save_drive is not None: #If we want to save in the drive
        torch.save(model.state_dict(), os.path.join(save_drive_dir, "best_model_loss.pt"))
        print(f"Model saved as best_model_loss.pt in drive")

    # Compute the metrics for the train and validation sets and append in the dictionary (both class AI/Real and category)
    train_accuracy_class,train_confusion_mat_class, train_f1_class = calculate_metrics_multihead(y_true_list = train_class_true_numpy, y_pred_list= train_class_pred_numpy, metrics = train_metrics, task="class")
    val_accuracy_class,val_confusion_mat_class, val_f1_class = calculate_metrics_multihead(y_true_list = val_class_true_numpy, y_pred_list= val_class_pred_numpy, metrics = val_metrics, task="class")
    train_accuracy_category,train_confusion_mat_category, train_f1_category = calculate_metrics_multihead(y_true_list = train_category_true_numpy, y_pred_list= train_category_pred_numpy, metrics = train_metrics, task="category")
    val_accuracy_category,val_confusion_mat_category, val_f1_category = calculate_metrics_multihead(y_true_list = val_category_true_numpy, y_pred_list= val_category_pred_numpy, metrics = val_metrics, task="category")

    train_metrics['combined_loss'].append(epoch_combined_loss)
    val_metrics['combined_loss'].append(epoch_combined_vloss)

    train_metrics['class_loss'].append(epoch_class_loss)
    val_metrics['class_loss'].append(val_epoch_class_loss)

    train_metrics['category_loss'].append(epoch_category_loss)
    val_metrics['category_loss'].append(val_epoch_category_loss)

    # If the class accuracy in the validation set is better in this epoch (w.r.t. the previous epochs), save the model
    if val_accuracy_class > best_vaccuracy_class:
      best_vaccuracy_class = val_accuracy_class
      torch.save(model.state_dict(), "best_model_class.pt")
      print(f"Model saved as best_model_class.pt")
      if save_in_drive and path_to_save_drive is not None: #If we want to save in the drive
        torch.save(model.state_dict(), os.path.join(save_drive_dir, "best_model_class.pt"))
        print(f"Model saved as best_model_class.pt in drive")

    # If the category accuracy in the validation set is better in this epoch (w.r.t. the previous epochs), save the model
    if val_accuracy_category > best_vaccuracy_category:
      best_vaccuracy_category = val_accuracy_category
      torch.save(model.state_dict(), "best_model_category.pt")
      print(f"Model saved as best_model_category.pt")
      if save_in_drive and path_to_save_drive is not None: #If we want to save in the drive
        torch.save(model.state_dict(), os.path.join(save_drive_dir, "best_model_category.pt"))
        print(f"Model saved as best_model_category.pt in drive")

    print(
      f"TRAIN\t\t"
      f"Combined Loss: {epoch_combined_loss:.4f}, "
      f"Class Loss: {epoch_class_loss:.4f}, "
      f"Category Loss: {epoch_category_loss:.4f}, "
      f"Accuracy Class: {train_accuracy_class:.4f}, "
      f"F1 Class: {train_f1_class:.4f}, "
      f"Accuracy Category: {train_accuracy_category:.4f}, "
      f"F1 Category: {train_f1_category:.4f}, "
    )

    print(
      f"VALIDATION\t"
      f"Combined Loss: {epoch_combined_vloss:.4f}, "
      f"Class Loss: {val_epoch_class_loss:.4f}, "
      f"Category Loss: {val_epoch_category_loss:.4f}, "
      f"Accuracy Class: {val_accuracy_class:.4f}, "
      f"F1 Class: {val_f1_class:.4f}, "
      f"Accuracy Category: {val_accuracy_category:.4f}, "
      f"F1 Category: {val_f1_category:.4f}, "
    )

    print("Train confusion matrix class:")
    print(train_confusion_mat_class)

    print("Train confusion matrix category:")
    print(train_confusion_mat_category)

    print("Validation confusion matrix class:")
    print(val_confusion_mat_class)

    print("Validation confusion matrix category:")
    print(val_confusion_mat_category)


  # Save the metrics in json
  with open("train_metrics.json", "w") as f:
    json.dump(train_metrics, f, indent=4)
  with open("val_metrics.json", "w") as f:
    json.dump(val_metrics, f, indent=4)

  if save_in_drive and path_to_save_drive is not None:
    with open(os.path.join(save_drive_dir, "train_metrics.json"), "w") as f:
      json.dump(train_metrics, f, indent=4)
    with open(os.path.join(save_drive_dir, "val_metrics.json"), "w") as f:
      json.dump(val_metrics, f, indent=4)
  print("Metrics of training and validation set saved in .json files!")

In [ ]:
#@title Load the checkpoint (DRCTConvB) previously trained for the Real/AI task, add a new head and fine-tune it to learn also the classification of the category.
"""
In this experiment:
1) We load the DRCTConvB previously trained with a single head (real/ai) and then we add and initialize the second head (category)
2) In the Phase 1-Fine-Tuning we train ONLY the new head (using only category_loss) for 4 epochs, in order to see if it can learn to correctly classify the categories from the embeddings learned for the Real/Ai Task.
3) In the Phase 2-Fine-Tuning we unlock all the pieces of the network and trained it with different learning rates accordingly to how we want to modify the features already learned for the first task;
For example, We assigned a low learning rate to the Head of the first task as it has already learned something,
while we go more violent on the shared-backbone in order to learn also features for classify the categories in the STEM and stages blocks.
In this second phase, we used a combined loss (class_loss and category_loss) with class_weight = 0.1 (as the network already learned this task) and category_weight = 1.0 to continue the previous work

"""
drct_rgb_core_multihead = DRCTConvB_MultiHead(
    checkpoint_path='/content/best_model_accuracy_base.pt',
    device=device
)
drct_rgb_core_multihead=drct_rgb_core_multihead.to(device)
optimizer = configure_fine_tuning_multihead_phase_1(drct_rgb_core_multihead)

train_multihead(
    train_loader = train_loader,
    val_loader = val_loader,
    model = drct_rgb_core_multihead,
    optimizer = optimizer,
    loss_fn_class = loss_fn,
    loss_fn_category = loss_fn,
    class_weight = 0.0,
    category_weight = 1.0,
    num_epochs = 4,
    device = device,
    path_to_save_drive = path_to_drive
)

optimizer= configure_fine_tuning_multihead_phase_2(drct_rgb_core_multihead)

train_multihead(
    train_loader = train_loader,
    val_loader = val_loader,
    model = drct_rgb_core_multihead,
    optimizer = optimizer,
    loss_fn_class = loss_fn,
    loss_fn_category = loss_fn,
    class_weight = 0.1,
    category_weight = 1.0,
    num_epochs = num_epochs,
    device = device,
    path_to_save_drive = path_to_drive
)

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(


Checkpoint loaded correctly.
EPOCH: 1/4


Validation Batches: 100%|██████████| 43/43 [00:19<00:00,  2.25it/s]


Model saved as best_model_loss.pt
Model saved as best_model_loss.pt in drive
Model saved as best_model_class.pt
Model saved as best_model_class.pt in drive
Model saved as best_model_category.pt
Model saved as best_model_category.pt in drive
TRAIN		Combined Loss: 1.0054, Class Loss: 0.1203, Category Loss: 1.0054, Accuracy Class: 0.9748, F1 Class: 0.9748, Accuracy Category: 0.3359, F1 Category: 0.3313, 
VALIDATION	Combined Loss: 0.9983, Class Loss: 0.7815, Category Loss: 0.9983, Accuracy Class: 0.7874, F1 Class: 0.7874, Accuracy Category: 0.3526, F1 Category: 0.3486, 
Train confusion matrix class:
[[3055   95]
 [  64 3086]]
Train confusion matrix category:
[[569 897 634]
 [542 947 611]
 [571 929 600]]
Validation confusion matrix class:
[[523 152]
 [135 540]]
Validation confusion matrix category:
[[155  87 208]
 [135 118 197]
 [142 105 203]]
EPOCH: 2/4


Validation Batches: 100%|██████████| 43/43 [00:18<00:00,  2.28it/s]


...

TRAIN		Combined Loss: 0.0965, Class Loss: 0.1078, Category Loss: 0.0954, Accuracy Class: 0.9714, F1 Class: 0.9714, Accuracy Category: 0.9571, F1 Category: 0.9572, 
VALIDATION	Combined Loss: 1.6312, Class Loss: 1.1332, Category Loss: 1.6810, Accuracy Class: 0.7859, F1 Class: 0.7849, Accuracy Category: 0.5289, F1 Category: 0.5293, 
Train confusion matrix class:
[[3057   93]
 [  87 3063]]
Train confusion matrix category:
[[1974   18  108]
 [  18 2067   15]
 [  99   12 1989]]
Validation confusion matrix class:
[[485 190]
 [ 99 576]]
Validation confusion matrix category:
[[213  60 177]
 [ 63 336  51]
 [243  42 165]]
EPOCH: 19/20


Validation Batches: 100%|██████████| 43/43 [00:18<00:00,  2.29it/s]


TRAIN		Combined Loss: 0.0823, Class Loss: 0.0828, Category Loss: 0.0822, Accuracy Class: 0.9808, F1 Class: 0.9808, Accuracy Category: 0.9638, F1 Category: 0.9638, 
VALIDATION	Combined Loss: 1.6075, Class Loss: 1.3769, Category Loss: 1.6305, Accuracy Class: 0.7637, F1 Class: 0.7607, Accuracy Category: 0.5415, F1 Category: 0.5335, 
Train confusion matrix class:
[[3094   56]
 [  65 3085]]
Train confusion matrix category:
[[1995   14   91]
 [  19 2073    8]
 [  90    6 2004]]
Validation confusion matrix class:
[[440 235]
 [ 84 591]]
Validation confusion matrix category:
[[195  76 179]
 [ 50 367  33]
 [214  67 169]]
EPOCH: 20/20


Validation Batches: 100%|██████████| 43/43 [00:18<00:00,  2.29it/s]

TRAIN		Combined Loss: 0.0673, Class Loss: 0.0781, Category Loss: 0.0662, Accuracy Class: 0.9814, F1 Class: 0.9814, Accuracy Category: 0.9667, F1 Category: 0.9666, 
VALIDATION	Combined Loss: 1.8966, Class Loss: 1.3906, Category Loss: 1.9472, Accuracy Class: 0.7800, F1 Class: 0.7798, Accuracy Category: 0.5267, F1 Category: 0.5145, 
Train confusion matrix class:
[[3101   49]
 [  68 3082]]
Train confusion matrix category:
[[1996   14   90]
 [  10 2086    4]
 [  85    7 2008]]
Validation confusion matrix class:
[[506 169]
 [128 547]]
Validation confusion matrix category:
[[191  91 168]
 [ 49 367  34]
 [215  82 153]]
Metrics of training and validation set saved in .json files!


In [ ]:
#@title Load the checkpoint (DRCTConvB_DFT) previously trained for the Real/AI task, add a new head and fine-tune it to learn also the classification of the category.

"""
In this experiment:
1) We load the DRCTConvB_DFT previously trained with a single head (real/ai) and then we add and initialize the second head (category)
2) In the Phase 1-Fine-Tuning we train ONLY the new head for 4 epochs, in order to see if it can learn to correctly classify the categories from the embeddings learned for the Real/Ai Task.
3) In the Phase 2-Fine-Tuning we unlock all the pieces of the network and trained it with different learning rates accordingly to how we want to modify the features already learned for the first task;
For example, We assigned a low learning rate to the Head of the first task as it has already learned something,
while we go more violent on the shared-backbone in order to learn also features for classify the categories in the STEM,STEM_DFT and stages blocks.
In this second phase, we used a combined loss (class_loss and category_loss) with class_weight = 0.1 (as the network already learned this task) and category_weight = 1.0 to continue the previous work

Then we compare the results with the previous experiment (DRCTConvB_MultiHead)

"""
drct_dft_multihead = DRCTConvB_DFT_MultiHead(
    personal_checkpoint_path='/content/best_model_accuracy_DFT.pt',
    device=device
)
drct_dft_multihead=drct_dft_multihead.to(device)
optimizer = configure_fine_tuning_multihead_phase_1(drct_dft_multihead)

train_multihead(
    train_loader = train_loader,
    val_loader = val_loader,
    model = drct_dft_multihead,
    optimizer = optimizer,
    loss_fn_class = loss_fn,
    loss_fn_category = loss_fn,
    class_weight = 0.0,
    category_weight = 1.0,
    num_epochs = 4,
    device = device,
    path_to_save_drive = path_to_drive
)

optimizer= configure_fine_tuning_multihead_phase_2(drct_dft_multihead)

train_multihead(
    train_loader = train_loader,
    val_loader = val_loader,
    model = drct_dft_multihead,
    optimizer = optimizer,
    loss_fn_class = loss_fn,
    loss_fn_category = loss_fn,
    class_weight = CLASS_WEIGHT,
    category_weight = CATEGORY_WEIGHT,
    num_epochs = num_epochs,
    device = device,
    path_to_save_drive = path_to_drive
)

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(


Personal checkpoint loaded correctly.
EPOCH: 1/4


Validation Batches: 100%|██████████| 43/43 [00:19<00:00,  2.23it/s]


Model saved as best_model_loss.pt
Model saved as best_model_loss.pt in drive
Model saved as best_model_class.pt
Model saved as best_model_class.pt in drive
Model saved as best_model_category.pt
Model saved as best_model_category.pt in drive
TRAIN		Combined Loss: 1.0094, Class Loss: 0.0252, Category Loss: 1.0094, Accuracy Class: 0.9956, F1 Class: 0.9956, Accuracy Category: 0.3354, F1 Category: 0.3318, 
VALIDATION	Combined Loss: 1.0007, Class Loss: 0.7932, Category Loss: 1.0007, Accuracy Class: 0.8163, F1 Class: 0.8163, Accuracy Category: 0.3519, F1 Category: 0.3489, 
Train confusion matrix class:
[[3134   16]
 [  12 3138]]
Train confusion matrix category:
[[753 818 529]
 [725 850 525]
 [721 869 510]]
Validation confusion matrix class:
[[547 128]
 [120 555]]
Validation confusion matrix category:
[[176 104 170]
 [159 119 172]
 [168 102 180]]
EPOCH: 2/4


Validation Batches: 100%|██████████| 43/43 [00:18<00:00,  2.27it/s]


...

Model saved as best_model_category.pt
Model saved as best_model_category.pt in drive
TRAIN		Combined Loss: 0.0497, Class Loss: 0.0572, Category Loss: 0.0489, Accuracy Class: 0.9871, F1 Class: 0.9871, Accuracy Category: 0.9848, F1 Category: 0.9848, 
VALIDATION	Combined Loss: 1.1452, Class Loss: 1.2589, Category Loss: 1.1338, Accuracy Class: 0.8133, F1 Class: 0.8132, Accuracy Category: 0.7363, F1 Category: 0.7376, 
Train confusion matrix class:
[[3113   37]
 [  44 3106]]
Train confusion matrix category:
[[2057    7   36]
 [   4 2091    5]
 [  40    4 2056]]
Validation confusion matrix class:
[[567 108]
 [144 531]]
Validation confusion matrix category:
[[294  39 117]
 [ 85 340  25]
 [ 77  13 360]]
EPOCH: 19/20


Validation Batches: 100%|██████████| 43/43 [00:18<00:00,  2.29it/s]


TRAIN		Combined Loss: 0.0727, Class Loss: 0.0905, Category Loss: 0.0710, Accuracy Class: 0.9752, F1 Class: 0.9752, Accuracy Category: 0.9759, F1 Category: 0.9759, 
VALIDATION	Combined Loss: 1.0012, Class Loss: 1.0173, Category Loss: 0.9996, Accuracy Class: 0.8000, F1 Class: 0.7998, Accuracy Category: 0.7015, F1 Category: 0.7040, 
Train confusion matrix class:
[[3076   74]
 [  82 3068]]
Train confusion matrix category:
[[2030   22   48]
 [  16 2076    8]
 [  49    9 2042]]
Validation confusion matrix class:
[[519 156]
 [114 561]]
Validation confusion matrix category:
[[282  29 139]
 [ 91 310  49]
 [ 85  10 355]]
EPOCH: 20/20


Validation Batches: 100%|██████████| 43/43 [00:18<00:00,  2.28it/s]

TRAIN		Combined Loss: 0.0646, Class Loss: 0.0743, Category Loss: 0.0637, Accuracy Class: 0.9827, F1 Class: 0.9827, Accuracy Category: 0.9781, F1 Category: 0.9781, 
VALIDATION	Combined Loss: 2.4521, Class Loss: 1.1320, Category Loss: 2.5842, Accuracy Class: 0.8067, F1 Class: 0.8065, Accuracy Category: 0.5726, F1 Category: 0.5361, 
Train confusion matrix class:
[[3099   51]
 [  58 3092]]
Train confusion matrix category:
[[2038   25   37]
 [  30 2066    4]
 [  38    4 2058]]
Validation confusion matrix class:
[[527 148]
 [113 562]]
Validation confusion matrix category:
[[ 92 323  35]
 [  6 442   2]
 [ 52 159 239]]
Metrics of training and validation set saved in .json files!


After finding out that DRCTConvB_MultiHead with the DFT STEM is better in both the two tasks, we continued experiments only with the New **Proposed model with DFT stem**

In [ ]:
#@title Load the checkpoint (DRCTConvB_DFT) previously trained for the Real/AI task, add a new head and start a new training phase ONLY for category transformation classification
"""
In this experiment:
1) We load the DRCTConvB previously trained with a single head (real/ai) and then we add and initialize the second head (category)
2) Then we start a new training phase on the whole model ONLY for category transformation classification using ONLY category loss.
We analyzed the results to see if the model can benefit from the embeddings learned for the Real/Ai Task
and if the category classification training can lead to unlearn the task of distinguish the Real/AI Images.

"""
drct_dft_multihead = DRCTConvB_DFT_MultiHead(
    personal_checkpoint_path='/content/best_model_accuracy_DFT.pt',
    device=device
)
drct_dft_multihead=drct_dft_multihead.to(device)
optimizer = torch.optim.Adam(drct_dft_multihead.parameters(), lr=lr)

train_multihead(
    train_loader = train_loader,
    val_loader = val_loader,
    model = drct_dft_multihead,
    optimizer = optimizer,
    loss_fn_class = loss_fn,
    loss_fn_category = loss_fn,
    class_weight = 0.0,
    category_weight = 1.0,
    num_epochs = 20,
    device = device,
    path_to_save_drive = path_to_drive
)

In [ ]:
#@title TRAIN DRCTConvB_DFT_MultiHead ONLY on classification category transformation task
"""
We trained the DFT network deactivating the first head (Real/AI) in order to see the performances that it can reach alone in the category classification task
"""
drct_dft_multihead = DRCTConvB_DFT_MultiHead(
    checkpoint_path=DRCT_CHECKPOINT_PATH,
    device=device
)
drct_dft_multihead=drct_dft_multihead.to(device)
optimizer = torch.optim.Adam(drct_dft_multihead.parameters(), lr=lr)

train_multihead(
    train_loader = train_loader,
    val_loader = val_loader,
    model = drct_dft_multihead,
    optimizer = optimizer,
    loss_fn_class = loss_fn,
    loss_fn_category = loss_fn,
    class_weight = 0.0,
    category_weight = 1.0,
    num_epochs = 20,
    device = device,
    path_to_save_drive = path_to_drive
)

In [ ]:
#@title TRAIN DRCTConvB_DFT_MultiHead with a combined loss of class_weight=1 and category_weight=0.25
drct_dft_multihead = DRCTConvB_DFT_MultiHead(
    checkpoint_path=DRCT_CHECKPOINT_PATH,
    device=device
)
drct_dft_multihead=drct_dft_multihead.to(device)
optimizer = torch.optim.AdamW(
    drct_dft_multihead.parameters(),
    lr=lr,
    weight_decay=1e-4
)

train_multihead(
    train_loader = train_loader,
    val_loader = val_loader,
    model = drct_dft_multihead,
    optimizer = optimizer,
    loss_fn_class = loss_fn,
    loss_fn_category = loss_fn,
    class_weight = 1.0,
    category_weight = 0.25,
    num_epochs = 20,
    device = device,
    path_to_save_drive = path_to_drive
)

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(


DCRT Checkpoint loaded correctly.
EPOCH: 1/20


Validation Batches: 100%|██████████| 43/43 [00:21<00:00,  1.99it/s]


Model saved as best_model_loss.pt
Model saved as best_model_loss.pt in drive
Model saved as best_model_class.pt
Model saved as best_model_class.pt in drive
Model saved as best_model_category.pt
Model saved as best_model_category.pt in drive
TRAIN		Combined Loss: 0.8376, Class Loss: 0.7886, Category Loss: 1.0339, Accuracy Class: 0.7363, F1 Class: 0.7363, Accuracy Category: 0.3549, F1 Category: 0.3552, 
VALIDATION	Combined Loss: 0.6001, Class Loss: 0.5028, Category Loss: 0.9890, Accuracy Class: 0.8467, F1 Class: 0.8453, Accuracy Category: 0.4215, F1 Category: 0.3722, 
Train confusion matrix class:
[[2368  782]
 [ 879 2271]]
Train confusion matrix category:
[[757 593 750]
 [663 770 667]
 [763 628 709]]
Validation confusion matrix class:
[[635  40]
 [167 508]]
Validation confusion matrix category:
[[214 212  24]
 [122 315  13]
 [202 208  40]]
EPOCH: 2/20


Validation Batches: 100%|██████████| 43/43 [00:19<00:00,  2.19it/s]


...

TRAIN		Combined Loss: 0.0049, Class Loss: 0.0001, Category Loss: 0.0240, Accuracy Class: 1.0000, F1 Class: 1.0000, Accuracy Category: 0.9930, F1 Category: 0.9930, 
VALIDATION	Combined Loss: 0.5677, Class Loss: 0.5996, Category Loss: 0.4402, Accuracy Class: 0.9096, F1 Class: 0.9096, Accuracy Category: 0.8852, F1 Category: 0.8854, 
Train confusion matrix class:
[[3150    0]
 [   0 3150]]
Train confusion matrix category:
[[2079    6   15]
 [   3 2097    0]
 [  19    1 2080]]
Validation confusion matrix class:
[[617  58]
 [ 64 611]]
Validation confusion matrix category:
[[379  28  43]
 [ 40 406   4]
 [ 38   2 410]]
EPOCH: 19/20


Validation Batches: 100%|██████████| 43/43 [00:19<00:00,  2.20it/s]


TRAIN		Combined Loss: 0.0116, Class Loss: 0.0092, Category Loss: 0.0209, Accuracy Class: 0.9976, F1 Class: 0.9976, Accuracy Category: 0.9924, F1 Category: 0.9924, 
VALIDATION	Combined Loss: 0.7257, Class Loss: 0.8002, Category Loss: 0.4279, Accuracy Class: 0.8763, F1 Class: 0.8756, Accuracy Category: 0.8393, F1 Category: 0.8403, 
Train confusion matrix class:
[[3142    8]
 [   7 3143]]
Train confusion matrix category:
[[2081    1   18]
 [   5 2095    0]
 [  20    4 2076]]
Validation confusion matrix class:
[[641  34]
 [133 542]]
Validation confusion matrix category:
[[381  43  26]
 [ 43 402   5]
 [ 88  12 350]]
EPOCH: 20/20


Validation Batches: 100%|██████████| 43/43 [00:19<00:00,  2.19it/s]

TRAIN		Combined Loss: 0.1000, Class Loss: 0.0874, Category Loss: 0.1505, Accuracy Class: 0.9790, F1 Class: 0.9790, Accuracy Category: 0.9405, F1 Category: 0.9405, 
VALIDATION	Combined Loss: 0.4534, Class Loss: 0.4670, Category Loss: 0.3986, Accuracy Class: 0.9052, F1 Class: 0.9052, Accuracy Category: 0.8422, F1 Category: 0.8442, 
Train confusion matrix class:
[[3087   63]
 [  69 3081]]
Train confusion matrix category:
[[1949   62   89]
 [  66 2007   27]
 [ 103   28 1969]]
Validation confusion matrix class:
[[617  58]
 [ 70 605]]
Validation confusion matrix category:
[[386  28  36]
 [ 64 381   5]
 [ 72   8 370]]
Metrics of training and validation saved into colab Files!


In [ ]:
#@title TRAIN DRCTConvB_DFT_MultiHead with a combined loss of class_weight=1 and category_weight=1
drct_dft_multihead = DRCTConvB_DFT_MultiHead(
    checkpoint_path=DRCT_CHECKPOINT_PATH,
    device=device
)
drct_dft_multihead=drct_dft_multihead.to(device)
optimizer = torch.optim.AdamW(
    drct_dft_multihead.parameters(),
    lr=lr,
    weight_decay=1e-4
)

train_multihead(
    train_loader = train_loader,
    val_loader = val_loader,
    model = drct_dft_multihead,
    optimizer = optimizer,
    loss_fn_class = loss_fn,
    loss_fn_category = loss_fn,
    class_weight = 1.0,
    category_weight = 1.0,
    num_epochs = 20,
    device = device,
    path_to_save_drive = path_to_drive
)

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(


DCRT Checkpoint loaded correctly.
EPOCH: 1/20


Validation Batches: 100%|██████████| 43/43 [00:19<00:00,  2.20it/s]


Model saved as best_model_loss.pt
Model saved as best_model_loss.pt in drive
Model saved as best_model_class.pt
Model saved as best_model_class.pt in drive
Model saved as best_model_category.pt
Model saved as best_model_category.pt in drive
TRAIN		Combined Loss: 0.9386, Class Loss: 0.8506, Category Loss: 1.0266, Accuracy Class: 0.7224, F1 Class: 0.7222, Accuracy Category: 0.3741, F1 Category: 0.3745, 
VALIDATION	Combined Loss: 0.8183, Class Loss: 0.5969, Category Loss: 1.0396, Accuracy Class: 0.8022, F1 Class: 0.7991, Accuracy Category: 0.3533, F1 Category: 0.2135, 
Train confusion matrix class:
[[2361  789]
 [ 960 2190]]
Train confusion matrix category:
[[759 554 787]
 [631 847 622]
 [741 608 751]]
Validation confusion matrix class:
[[625  50]
 [217 458]]
Validation confusion matrix category:
[[ 33 417   0]
 [  6 444   0]
 [ 35 415   0]]
EPOCH: 2/20


Validation Batches: 100%|██████████| 43/43 [00:19<00:00,  2.22it/s]


...

TRAIN		Combined Loss: 0.0184, Class Loss: 0.0165, Category Loss: 0.0203, Accuracy Class: 0.9963, F1 Class: 0.9963, Accuracy Category: 0.9916, F1 Category: 0.9916, 
VALIDATION	Combined Loss: 0.6169, Class Loss: 0.8025, Category Loss: 0.4313, Accuracy Class: 0.8911, F1 Class: 0.8906, Accuracy Category: 0.8763, F1 Category: 0.8782, 
Train confusion matrix class:
[[3139   11]
 [  12 3138]]
Train confusion matrix category:
[[2074    4   22]
 [   4 2095    1]
 [  18    4 2078]]
Validation confusion matrix class:
[[649  26]
 [121 554]]
Validation confusion matrix category:
[[412  13  25]
 [ 51 395   4]
 [ 71   3 376]]
EPOCH: 19/20


Validation Batches: 100%|██████████| 43/43 [00:19<00:00,  2.23it/s]


TRAIN		Combined Loss: 0.0246, Class Loss: 0.0260, Category Loss: 0.0232, Accuracy Class: 0.9938, F1 Class: 0.9938, Accuracy Category: 0.9917, F1 Category: 0.9917, 
VALIDATION	Combined Loss: 0.5807, Class Loss: 0.7785, Category Loss: 0.3830, Accuracy Class: 0.8756, F1 Class: 0.8754, Accuracy Category: 0.8881, F1 Category: 0.8879, 
Train confusion matrix class:
[[3131   19]
 [  20 3130]]
Train confusion matrix category:
[[2074    3   23]
 [   7 2092    1]
 [  17    1 2082]]
Validation confusion matrix class:
[[567 108]
 [ 60 615]]
Validation confusion matrix category:
[[372  28  50]
 [ 36 408   6]
 [ 28   3 419]]
EPOCH: 20/20


Validation Batches: 100%|██████████| 43/43 [00:19<00:00,  2.22it/s]

TRAIN		Combined Loss: 0.0250, Class Loss: 0.0100, Category Loss: 0.0401, Accuracy Class: 0.9976, F1 Class: 0.9976, Accuracy Category: 0.9851, F1 Category: 0.9851, 
VALIDATION	Combined Loss: 0.6124, Class Loss: 0.7218, Category Loss: 0.5029, Accuracy Class: 0.9074, F1 Class: 0.9071, Accuracy Category: 0.8622, F1 Category: 0.8643, 
Train confusion matrix class:
[[3142    8]
 [   7 3143]]
Train confusion matrix category:
[[2059   24   17]
 [  21 2077    2]
 [  26    4 2070]]
Validation confusion matrix class:
[[651  24]
 [101 574]]
Validation confusion matrix category:
[[402  10  38]
 [ 84 363   3]
 [ 49   2 399]]
Metrics of training and validation saved into colab Files!


In [ ]:
#@title TRAIN DRCTConvB_DFT_MultiHead with a combined loss of class_weight=1 and category_weight=1.75
drct_dft_multihead = DRCTConvB_DFT_MultiHead(
    checkpoint_path=DRCT_CHECKPOINT_PATH,
    device=device
)
drct_dft_multihead=drct_dft_multihead.to(device)
optimizer = torch.optim.AdamW(
    drct_dft_multihead.parameters(),
    lr=lr,
    weight_decay=1e-4
)

train_multihead(
    train_loader = train_loader,
    val_loader = val_loader,
    model = drct_dft_multihead,
    optimizer = optimizer,
    loss_fn_class = loss_fn,
    loss_fn_category = loss_fn,
    class_weight = 1.0,
    category_weight = 1.75,
    num_epochs = 20,
    device = device,
    path_to_save_drive = path_to_drive
)

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(


DCRT Checkpoint loaded correctly.
EPOCH: 1/20


Validation Batches: 100%|██████████| 43/43 [00:19<00:00,  2.23it/s]


Model saved as best_model_loss.pt
Model saved as best_model_loss.pt in drive
Model saved as best_model_class.pt
Model saved as best_model_class.pt in drive
Model saved as best_model_category.pt
Model saved as best_model_category.pt in drive
TRAIN		Combined Loss: 0.9493, Class Loss: 0.8681, Category Loss: 0.9957, Accuracy Class: 0.7179, F1 Class: 0.7178, Accuracy Category: 0.3963, F1 Category: 0.3962, 
VALIDATION	Combined Loss: 0.7411, Class Loss: 0.5891, Category Loss: 0.8280, Accuracy Class: 0.8096, F1 Class: 0.8049, Accuracy Category: 0.5496, F1 Category: 0.5133, 
Train confusion matrix class:
[[2322  828]
 [ 949 2201]]
Train confusion matrix category:
[[807 565 728]
 [567 951 582]
 [776 585 739]]
Validation confusion matrix class:
[[652  23]
 [234 441]]
Validation confusion matrix category:
[[215 165  70]
 [ 21 417  12]
 [146 194 110]]
EPOCH: 2/20


Validation Batches: 100%|██████████| 43/43 [00:18<00:00,  2.30it/s]


...

Model saved as best_model_category.pt
Model saved as best_model_category.pt in drive
TRAIN		Combined Loss: 0.0267, Class Loss: 0.0325, Category Loss: 0.0234, Accuracy Class: 0.9916, F1 Class: 0.9916, Accuracy Category: 0.9908, F1 Category: 0.9908, 
VALIDATION	Combined Loss: 0.4391, Class Loss: 0.5398, Category Loss: 0.3817, Accuracy Class: 0.9111, F1 Class: 0.9111, Accuracy Category: 0.9037, F1 Category: 0.9044, 
Train confusion matrix class:
[[3124   26]
 [  27 3123]]
Train confusion matrix category:
[[2080    5   15]
 [  10 2086    4]
 [  22    2 2076]]
Validation confusion matrix class:
[[601  74]
 [ 46 629]]
Validation confusion matrix category:
[[405   6  39]
 [ 43 397  10]
 [ 31   1 418]]
EPOCH: 19/20


Validation Batches: 100%|██████████| 43/43 [00:19<00:00,  2.26it/s]


TRAIN		Combined Loss: 0.0217, Class Loss: 0.0247, Category Loss: 0.0200, Accuracy Class: 0.9941, F1 Class: 0.9941, Accuracy Category: 0.9919, F1 Category: 0.9919, 
VALIDATION	Combined Loss: 0.5537, Class Loss: 0.5679, Category Loss: 0.5456, Accuracy Class: 0.9126, F1 Class: 0.9125, Accuracy Category: 0.8726, F1 Category: 0.8715, 
Train confusion matrix class:
[[3130   20]
 [  17 3133]]
Train confusion matrix category:
[[2079    8   13]
 [  11 2088    1]
 [  16    2 2082]]
Validation confusion matrix class:
[[633  42]
 [ 76 599]]
Validation confusion matrix category:
[[355  64  31]
 [ 14 433   3]
 [ 45  15 390]]
EPOCH: 20/20


Validation Batches: 100%|██████████| 43/43 [00:18<00:00,  2.30it/s]


Model saved as best_model_category.pt
Model saved as best_model_category.pt in drive
TRAIN		Combined Loss: 0.0178, Class Loss: 0.0063, Category Loss: 0.0244, Accuracy Class: 0.9981, F1 Class: 0.9981, Accuracy Category: 0.9910, F1 Category: 0.9910, 
VALIDATION	Combined Loss: 0.5031, Class Loss: 0.7284, Category Loss: 0.3744, Accuracy Class: 0.8963, F1 Class: 0.8959, Accuracy Category: 0.9044, F1 Category: 0.9047, 
Train confusion matrix class:
[[3144    6]
 [   6 3144]]
Train confusion matrix category:
[[2076    6   18]
 [   8 2090    2]
 [  21    2 2077]]
Validation confusion matrix class:
[[646  29]
 [111 564]]
Validation confusion matrix category:
[[396  30  24]
 [ 24 426   0]
 [ 47   4 399]]
Metrics of training and validation saved into colab Files!


In [ ]:
#@title TRAIN DRCTConvB_DFT_MultiHead with a combined loss of class_weight=1 and category_weight=2.7665
drct_dft_multihead = DRCTConvB_DFT_MultiHead(
    checkpoint_path=DRCT_CHECKPOINT_PATH,
    device=device
)
drct_dft_multihead=drct_dft_multihead.to(device)
optimizer = torch.optim.AdamW(
    drct_dft_multihead.parameters(),
    lr=lr,
    weight_decay=1e-4
)

train_multihead(
    train_loader = train_loader,
    val_loader = val_loader,
    model = drct_dft_multihead,
    optimizer = optimizer,
    loss_fn_class = loss_fn,
    loss_fn_category = loss_fn,
    class_weight = 1.0,
    category_weight = 2.7665,
    num_epochs = 20,
    device = device,
    path_to_save_drive = path_to_drive
)

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(


DCRT Checkpoint loaded correctly.
EPOCH: 1/20


Validation Batches: 100%|██████████| 43/43 [00:05<00:00,  8.35it/s]


Model saved as best_model_loss.pt
Model saved as best_model_loss.pt in drive
Model saved as best_model_class.pt
Model saved as best_model_class.pt in drive
Model saved as best_model_category.pt
Model saved as best_model_category.pt in drive
TRAIN		Combined Loss: 0.9979, Class Loss: 0.9703, Category Loss: 1.0079, Accuracy Class: 0.6600, F1 Class: 0.6600, Accuracy Category: 0.3821, F1 Category: 0.3829, 
VALIDATION	Combined Loss: 0.9012, Class Loss: 0.8094, Category Loss: 0.9344, Accuracy Class: 0.6867, F1 Class: 0.6580, Accuracy Category: 0.4681, F1 Category: 0.3632, 
Train confusion matrix class:
[[2058 1092]
 [1050 2100]]
Train confusion matrix category:
[[799 514 787]
 [613 810 677]
 [777 525 798]]
Validation confusion matrix class:
[[659  16]
 [407 268]]
Validation confusion matrix category:
[[199 250   1]
 [ 18 432   0]
 [194 255   1]]
EPOCH: 2/20


Validation Batches: 100%|██████████| 43/43 [00:03<00:00, 13.50it/s]


...

TRAIN		Combined Loss: 0.0138, Class Loss: 0.0204, Category Loss: 0.0115, Accuracy Class: 0.9956, F1 Class: 0.9956, Accuracy Category: 0.9960, F1 Category: 0.9960, 
VALIDATION	Combined Loss: 0.4302, Class Loss: 0.6108, Category Loss: 0.3650, Accuracy Class: 0.9141, F1 Class: 0.9141, Accuracy Category: 0.8941, F1 Category: 0.8945, 
Train confusion matrix class:
[[3138   12]
 [  16 3134]]
Train confusion matrix category:
[[2088    3    9]
 [   5 2094    1]
 [   7    0 2093]]
Validation confusion matrix class:
[[628  47]
 [ 69 606]]
Validation confusion matrix category:
[[390  27  33]
 [ 37 412   1]
 [ 40   5 405]]
EPOCH: 19/20


Validation Batches: 100%|██████████| 43/43 [00:03<00:00, 13.51it/s]


TRAIN		Combined Loss: 0.0315, Class Loss: 0.0266, Category Loss: 0.0333, Accuracy Class: 0.9932, F1 Class: 0.9932, Accuracy Category: 0.9876, F1 Category: 0.9876, 
VALIDATION	Combined Loss: 0.3643, Class Loss: 0.5354, Category Loss: 0.3025, Accuracy Class: 0.9022, F1 Class: 0.9020, Accuracy Category: 0.8993, F1 Category: 0.8993, 
Train confusion matrix class:
[[3130   20]
 [  23 3127]]
Train confusion matrix category:
[[2064   17   19]
 [  16 2078    6]
 [  16    4 2080]]
Validation confusion matrix class:
[[580  95]
 [ 37 638]]
Validation confusion matrix category:
[[386  40  24]
 [ 28 420   2]
 [ 35   7 408]]
EPOCH: 20/20


Validation Batches: 100%|██████████| 43/43 [00:03<00:00, 13.52it/s]

TRAIN		Combined Loss: 0.0182, Class Loss: 0.0173, Category Loss: 0.0185, Accuracy Class: 0.9962, F1 Class: 0.9962, Accuracy Category: 0.9938, F1 Category: 0.9938, 
VALIDATION	Combined Loss: 0.5070, Class Loss: 0.5824, Category Loss: 0.4797, Accuracy Class: 0.9170, F1 Class: 0.9169, Accuracy Category: 0.8711, F1 Category: 0.8721, 
Train confusion matrix class:
[[3136   14]
 [  10 3140]]
Train confusion matrix category:
[[2083    7   10]
 [   4 2096    0]
 [  16    2 2082]]
Validation confusion matrix class:
[[643  32]
 [ 80 595]]
Validation confusion matrix category:
[[395   5  50]
 [ 82 362   6]
 [ 29   2 419]]
Metrics of training and validation set saved in .json files!


In [ ]:
#@title TRAIN DRCTConvB_DFT_MultiHead with a combined loss of class_weight=1 and category_weight=5
drct_dft_multihead = DRCTConvB_DFT_MultiHead(
    checkpoint_path=DRCT_CHECKPOINT_PATH,
    device=device
)
drct_dft_multihead=drct_dft_multihead.to(device)
optimizer = torch.optim.AdamW(
    drct_dft_multihead.parameters(),
    lr=lr,
    weight_decay=1e-4
)

train_multihead(
    train_loader = train_loader,
    val_loader = val_loader,
    model = drct_dft_multihead,
    optimizer = optimizer,
    loss_fn_class = loss_fn,
    loss_fn_category = loss_fn,
    class_weight = 1.0,
    category_weight = 5.0,
    num_epochs = 20,
    device = device,
    path_to_save_drive = path_to_drive
)

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(


DCRT Checkpoint loaded correctly.
EPOCH: 1/20


Validation Batches: 100%|██████████| 43/43 [00:19<00:00,  2.25it/s]


Model saved as best_model_loss.pt
Model saved as best_model_loss.pt in drive
Model saved as best_model_class.pt
Model saved as best_model_class.pt in drive
Model saved as best_model_category.pt
Model saved as best_model_category.pt in drive
TRAIN		Combined Loss: 1.0770, Class Loss: 1.2036, Category Loss: 1.0517, Accuracy Class: 0.5075, F1 Class: 0.5075, Accuracy Category: 0.3410, F1 Category: 0.3405, 
VALIDATION	Combined Loss: 1.0333, Class Loss: 1.0345, Category Loss: 1.0330, Accuracy Class: 0.5015, F1 Class: 0.3392, Accuracy Category: 0.3237, F1 Category: 0.2540, 
Train confusion matrix class:
[[1591 1559]
 [1544 1606]]
Train confusion matrix category:
[[780 687 633]
 [768 710 622]
 [736 706 658]]
Validation confusion matrix class:
[[  4 671]
 [  2 673]]
Validation confusion matrix category:
[[  0 281 169]
 [  0 279 171]
 [  0 292 158]]
EPOCH: 2/20


Validation Batches: 100%|██████████| 43/43 [00:18<00:00,  2.30it/s]


...

Model saved as best_model_class.pt
Model saved as best_model_class.pt in drive
TRAIN		Combined Loss: 0.0188, Class Loss: 0.0280, Category Loss: 0.0170, Accuracy Class: 0.9930, F1 Class: 0.9930, Accuracy Category: 0.9938, F1 Category: 0.9938, 
VALIDATION	Combined Loss: 0.4666, Class Loss: 0.6685, Category Loss: 0.4262, Accuracy Class: 0.8978, F1 Class: 0.8978, Accuracy Category: 0.9096, F1 Category: 0.9105, 
Train confusion matrix class:
[[3128   22]
 [  22 3128]]
Train confusion matrix category:
[[2085   11    4]
 [  11 2086    3]
 [   9    1 2090]]
Validation confusion matrix class:
[[609  66]
 [ 72 603]]
Validation confusion matrix category:
[[413  11  26]
 [ 50 400   0]
 [ 32   3 415]]
EPOCH: 19/20


Validation Batches: 100%|██████████| 43/43 [00:18<00:00,  2.30it/s]


TRAIN		Combined Loss: 0.0296, Class Loss: 0.0345, Category Loss: 0.0287, Accuracy Class: 0.9913, F1 Class: 0.9913, Accuracy Category: 0.9910, F1 Category: 0.9910, 
VALIDATION	Combined Loss: 0.3924, Class Loss: 0.7834, Category Loss: 0.3142, Accuracy Class: 0.8896, F1 Class: 0.8896, Accuracy Category: 0.9133, F1 Category: 0.9131, 
Train confusion matrix class:
[[3125   25]
 [  30 3120]]
Train confusion matrix category:
[[2076    6   18]
 [   4 2095    1]
 [  27    1 2072]]
Validation confusion matrix class:
[[616  59]
 [ 90 585]]
Validation confusion matrix category:
[[388  25  37]
 [ 33 417   0]
 [ 20   2 428]]
EPOCH: 20/20


Validation Batches: 100%|██████████| 43/43 [00:18<00:00,  2.28it/s]


Model saved as best_model_class.pt
Model saved as best_model_class.pt in drive
TRAIN		Combined Loss: 0.0167, Class Loss: 0.0258, Category Loss: 0.0148, Accuracy Class: 0.9949, F1 Class: 0.9949, Accuracy Category: 0.9948, F1 Category: 0.9948, 
VALIDATION	Combined Loss: 0.4382, Class Loss: 0.6757, Category Loss: 0.3907, Accuracy Class: 0.8993, F1 Class: 0.8992, Accuracy Category: 0.9067, F1 Category: 0.9066, 
Train confusion matrix class:
[[3133   17]
 [  15 3135]]
Train confusion matrix category:
[[2085    2   13]
 [   3 2097    0]
 [  14    1 2085]]
Validation confusion matrix class:
[[622  53]
 [ 83 592]]
Validation confusion matrix category:
[[383  16  51]
 [ 35 414   1]
 [ 22   1 427]]
Metrics of training and validation saved into colab Files!


#Test

##Single Head

In [ ]:
#@title Test Loop SIngle Head
def test_singlehead(model, test_dataloader, loss_fn, device):

  model.eval()

  # Definition of the metrics dictionary
  test_metrics = {
    'accuracy': [],
    'confusion_mat': [],
    'f1': [],
    'loss': [],
  }

  test_loss = 0
  num_samples = 0
  y_true_list = []
  y_pred_list = []
  categories_list= []

  with torch.no_grad():
    for batch in tqdm(test_dataloader, desc="Testing..."):
      rgb,dft, y_true, cats = batch
      if LAZY_LOAD:
        rgb = rgb.to(device,non_blocking=True)
        y_true = y_true.to(device,non_blocking=True)
      if isinstance(model,DRCTConvB):
        logits = model(rgb)
      elif isinstance(model,DRCTConvB_DFT):
        if LAZY_LOAD:
          dft = dft.to(device,non_blocking=True)
        logits = model(rgb,dft)
      else:
        print("Model not supported!")


      loss = loss_fn(logits, y_true)
      y_pred= logits.argmax(dim=1)

      y_true_list.append(y_true.cpu().detach())
      y_pred_list.append(y_pred.cpu().detach())
      categories_list.append(cats.cpu().detach())

      batch_size = y_true.size(0)
      test_loss += loss.item() * batch_size
      num_samples += batch_size

  # Calculate the average loss over all batches
  test_loss_avg = test_loss / num_samples

  test_y_true_tensor = torch.cat(y_true_list, dim=0)
  test_y_pred_tensor = torch.cat(y_pred_list, dim=0)
  test_categories_tensor = torch.cat(categories_list, dim=0)

  test_y_true_numpy = test_y_true_tensor.numpy()
  test_y_pred_numpy = test_y_pred_tensor.numpy()
  test_categories_numpy = test_categories_tensor.numpy()

  #Create tables for visualizing real/Ai detection accuracy separately for each category transformation
  dataframe_complete_analysis = pd.DataFrame({
  "y_true": test_y_true_numpy,
  "y_pred": test_y_pred_numpy,
  "category_true": test_categories_numpy})

  dataframe_complete_analysis["transformation"] = dataframe_complete_analysis["category_true"].map(category_mapping)

  dataframe_complete_analysis["class"] = dataframe_complete_analysis["y_true"].map(class_mapping)

  dataframe_complete_analysis["predicted_class"] = dataframe_complete_analysis["y_pred"].map(class_mapping)

  dataframe_complete_analysis["correct"] = (
    dataframe_complete_analysis["y_true"] == dataframe_complete_analysis["y_pred"]
  ).astype(int)

  accuracy_by_category_and_class = (
    dataframe_complete_analysis
    .groupby(["transformation", "class"])
    .agg(
      accuracy=("correct", "mean"),
      num_samples=("correct", "size"))
    .reset_index())

  accuracy_by_category_and_class["accuracy"] *= 100

  print(accuracy_by_category_and_class)

  accuracy_table = dataframe_complete_analysis.pivot_table(
    index="transformation",
    columns="class",
    values="correct",
    aggfunc="mean") * 100

  accuracy_table["overall"] = (
    dataframe_complete_analysis
    .groupby("transformation")["correct"]
    .mean() * 100)

  print(accuracy_table.round(2))

  # Calculate the metrics for the test set
  accuracy,confusion_mat,f1 = calculate_metrics(y_true_list = test_y_true_numpy, y_pred_list= test_y_pred_numpy, metrics = test_metrics)
  test_metrics['loss'].append(test_loss_avg)


  print(
    f"TEST\t\t"
    f"Loss: {test_loss_avg:.4f}, "
    f"Accuracy: {accuracy:.4f}, "
    f"F1: {f1:.4f}, ")

  print("Test confusion matrix:")
  print(confusion_mat)

  # Save the metrics in json
  with open("test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=4)
  print("Metrics test set saved in .json files!")

In [ ]:
#@title Load and test the model
model_path = '/content/best_model_accuracy_DFT.pt'
best_model = DRCTConvB_DFT()

best_model = best_model.to(device)
best_model.load_state_dict(torch.load(model_path,map_location=device,weights_only=True),strict=True)

test_singlehead(
    model = best_model,
    test_dataloader = test_loader,
    loss_fn = loss_fn,
    device = device
)

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(
Testing...: 100%|██████████| 43/43 [00:47<00:00,  1.11s/it]

  transformation class   accuracy  num_samples
0       original    ai  90.666667          225
1       original  real  70.666667          225
2      redigital    ai  90.222222          225
3      redigital  real  76.888889          225
4       transfer    ai  91.111111          225
5       transfer  real  68.444444          225
class              ai   real  overall
transformation                       
original        90.67  70.67    80.67
redigital       90.22  76.89    83.56
transfer        91.11  68.44    79.78
TEST		Loss: 0.5921, Accuracy: 0.8133, F1: 0.8117, 
Test confusion matrix:
[[486 189]
 [ 63 612]]
Metrics test set saved in .json files!


##Multi Head

In [ ]:
#@title TEST Loop MULTIHEAD
def test_multihead(model, test_dataloader, loss_fn_class,loss_fn_category,class_weight,category_weight, device):

  model.eval()

  # Definition of the metrics dictionary
  test_metrics = {
    'accuracy_class': [],
    'f1_class':[],
    'confusion_mat_class': [],
    'accuracy_category': [],
    'f1_category': [],
    'confusion_mat_category': [],
    'combined_loss': [],
    'class_loss': [],
    'category_loss': [],
  }

  test_combined_loss_sum = 0.0
  test_class_loss_sum = 0.0
  test_category_loss_sum = 0.0
  num_samples = 0
  class_true_list = []
  class_pred_list = []
  category_true_list = []
  category_pred_list = []

  with torch.no_grad():
    for batch in tqdm(test_dataloader, desc="Testing..."):
      rgb,dft, class_true, category_true = batch
      if LAZY_LOAD:
        rgb = rgb.to(device,non_blocking=True)
        class_true = class_true.to(device,non_blocking=True)
        category_true = category_true.to(device,non_blocking=True)
      if isinstance(model,DRCTConvB_MultiHead):
        logits_class, logits_category = model(rgb)
      elif isinstance(model,DRCTConvB_DFT_MultiHead):
        if LAZY_LOAD:
          dft = dft.to(device,non_blocking=True)
        logits_class, logits_category = model(rgb,dft)
      else:
        print("Model not supported!")

      loss_class = loss_fn_class(logits_class, class_true) / math.log(2) #Normlization of the binary Real/AI classfication loss by the uniform-prediction log(2)
      loss_category = loss_fn_category(logits_category, category_true) / math.log(3) #Normlization of the three category transformation classfication loss by the uniform-prediction log(3)
      loss = ((class_weight*loss_class) + (category_weight * loss_category))/(class_weight + category_weight) #Weighted combined loss of class AI/Real loss and category transformation loss

      class_pred= logits_class.argmax(dim=1)
      class_true_list.append(class_true.cpu().detach())
      class_pred_list.append(class_pred.cpu().detach())

      category_pred= logits_category.argmax(dim=1)
      category_true_list.append(category_true.cpu().detach())
      category_pred_list.append(category_pred.cpu().detach())

      batch_size = class_true.size(0)
      test_combined_loss_sum += loss.item() * batch_size
      test_class_loss_sum += loss_class.item() * batch_size
      test_category_loss_sum += loss_category.item() * batch_size
      num_samples += batch_size

  # Calculate the average losses over all batches
  test_combined_loss_avg = test_combined_loss_sum / num_samples
  test_class_loss_avg = test_class_loss_sum / num_samples
  test_category_loss_avg = test_category_loss_sum / num_samples


  test_metrics['combined_loss'].append(test_combined_loss_avg)
  test_metrics['class_loss'].append(test_class_loss_avg)
  test_metrics['category_loss'].append(test_category_loss_avg)



  test_class_true_tensor = torch.cat(class_true_list, dim=0)
  test_class_pred_tensor = torch.cat(class_pred_list, dim=0)
  test_category_true_tensor = torch.cat(category_true_list, dim=0)
  test_category_pred_tensor = torch.cat(category_pred_list, dim=0)


  test_class_true_numpy = test_class_true_tensor.numpy()
  test_class_pred_numpy = test_class_pred_tensor.numpy()
  test_category_true_numpy = test_category_true_tensor.numpy()
  test_category_pred_numpy = test_category_pred_tensor.numpy()


  #Calculate the metrics for the test set (both class AI/Real and category) and append in the dictionary
  test_accuracy_class,test_confusion_mat_class, test_f1_class = calculate_metrics_multihead(y_true_list = test_class_true_numpy, y_pred_list= test_class_pred_numpy, metrics = test_metrics, task="class")
  test_accuracy_category,test_confusion_mat_category, test_f1_category = calculate_metrics_multihead(y_true_list = test_category_true_numpy, y_pred_list= test_category_pred_numpy, metrics = test_metrics, task="category")


  #Create tables for visualizing real/Ai detection accuracy separately for each category transformation
  dataframe_complete_analysis = pd.DataFrame({
    "class_true": test_class_true_numpy,
    "class_pred": test_class_pred_numpy,
    "category_true": test_category_true_numpy,
    "category_pred": test_category_pred_numpy})

  dataframe_complete_analysis["transformation"] = dataframe_complete_analysis["category_true"].map(category_mapping)
  dataframe_complete_analysis["predicted_transformation"] = (dataframe_complete_analysis["category_pred"].map(category_mapping))

  dataframe_complete_analysis["class"] = dataframe_complete_analysis["class_true"].map(class_mapping)

  dataframe_complete_analysis["predicted_class"] = dataframe_complete_analysis["class_pred"].map(class_mapping)

  dataframe_complete_analysis["class_correct"] = (
    dataframe_complete_analysis["class_true"] == dataframe_complete_analysis["class_pred"]
  ).astype(int)

  dataframe_complete_analysis["transformation_correct"] = (
    dataframe_complete_analysis["category_true"] == dataframe_complete_analysis["category_pred"]
  ).astype(int)

  accuracy_by_category_and_class = (
    dataframe_complete_analysis
    .groupby(["transformation", "class"]) # Calculating separately the accuracy prediction label_AI for each of the 6 big family (original/ai, original/real, redigital/ai etc.)
    .agg(
      accuracy=("class_correct", "mean"),
      num_samples=("class_correct", "size"))
    .reset_index())

  accuracy_by_category_and_class["accuracy"] *= 100

  print(accuracy_by_category_and_class.round(2).to_string(index=False))

  accuracy_table = dataframe_complete_analysis.pivot_table(
    index="transformation",
    columns="class",
    values="class_correct",
    aggfunc="mean") * 100

  accuracy_table["overall"] = (
    dataframe_complete_analysis
    .groupby("transformation")["class_correct"]
    .mean() * 100)

  print(accuracy_table.round(2))


  print(
    f"TEST\t"
    f"Combined Loss: {test_combined_loss_avg:.4f}, "
    f"Class Loss: {test_class_loss_avg:.4f}, "
    f"Category Loss: {test_category_loss_avg:.4f}, "
    f"Accuracy Class: {test_accuracy_class:.4f}, "
    f"F1 Class: {test_f1_class:.4f}, "
    f"Accuracy Category: {test_accuracy_category:.4f}, "
    f"F1 Category: {test_f1_category:.4f}, ")

  print("Test confusion matrix class:")
  print(test_confusion_mat_class)

  print("Test confusion matrix category:")
  print(test_confusion_mat_category)

  # Save the metrics in json
  with open("test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=4)
  print("Metrics of test set saved in .json files!")

In [ ]:
#@title Load and test the model
model_path = '/content/best_model_class.pt'
best_model = DRCTConvB_DFT_MultiHead()

best_model = best_model.to(device)
best_model.load_state_dict(torch.load(model_path,map_location=device,weights_only=True),strict=True)

test_multihead(
    model = best_model,
    test_dataloader = test_loader,
    loss_fn_class= loss_fn,
    loss_fn_category= loss_fn,
    class_weight = CLASS_WEIGHT,
    category_weight = CATEGORY_WEIGHT,
    device = device
)

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(
Testing...: 100%|██████████| 43/43 [00:03<00:00, 13.71it/s]

transformation class  accuracy  num_samples
      original    ai     93.33          225
      original  real     92.89          225
     redigital    ai     97.78          225
     redigital  real     87.11          225
      transfer    ai     95.11          225
      transfer  real     88.89          225
class              ai   real  overall
transformation                       
original        93.33  92.89    93.11
redigital       97.78  87.11    92.44
transfer        95.11  88.89    92.00
TEST	Combined Loss: 0.3509, Class Loss: 0.4153, Category Loss: 0.3276, Accuracy Class: 0.9252, F1 Class: 0.9251, Accuracy Category: 0.9022, F1 Category: 0.9019, 
Test confusion matrix class:
[[605  70]
 [ 31 644]]
Test confusion matrix category:
[[388  22  40]
 [ 13 436   1]
 [ 50   6 394]]
Metrics of test set saved in .json files!
